#### N10 — Full Evaluation (M0–M3)

Evaluates all four peer identification models on every valuation multiple
and k. Produces hypothesis tests (H1–H3), annual/sector diagnostics,
robustness checks (Sections 8.3–8.6), and case studies.

**Input:** `peers_m0–m3.parquet` (N5–N9), `multiples.parquet` (N4), `panel_clean.parquet` (N1)
**Output:** `results_main.csv`, figures, robustness tables

### 0 · Setup

In [ ]:
# imports & config
import sys
from pathlib import Path

notebook_dir = Path('__file__').parent if '__file__' in dir() else Path.cwd()
repo_root = next(
    (p for p in [notebook_dir, *notebook_dir.parents] if (p / 'config.py').exists()),
    None
)
if repo_root is None:
    raise FileNotFoundError('config.py not found — check repo structure')
sys.path.insert(0, str(repo_root))

from config import *
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import wilcoxon
import json
import warnings
warnings.filterwarnings('ignore')

np.random.seed(RANDOM_SEED)

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.edgecolor': '#333333', 'axes.linewidth': 0.8,
    'axes.grid': True, 'grid.color': '#e5e5e5', 'grid.linewidth': 0.6,
    'font.family': 'serif', 'font.size': 10,
    'axes.titlesize': 11, 'axes.labelsize': 10,
    'legend.fontsize': 9, 'savefig.dpi': FIGURE_DPI,
    'savefig.bbox': 'tight', 'savefig.facecolor': 'white',
})

C0, C1, C2, C3 = '#2166ac', '#d6604d', '#4dac26', '#8073ac'
CGREY = '#888888'
C = dict(MODEL_COLORS)

print('Config loaded.')
print(f'  PEERS_M1   : {PEERS_M1}')
print(f'  PEERS_M3   : {PEERS_M3}')
print(f'  MULTIPLES  : {MULTIPLES}')
print(f'  RESULTS    : {RESULTS_MAIN}')

In [ ]:
INPUTS  = [PEERS_M0, PEERS_M1, PEERS_M2, PEERS_M3, PEERS_M3_RRF, MULTIPLES, PANEL_CLEAN]
OUTPUTS = [
    RESULTS_MAIN,
    DATA_RESULTS / 'mape_robustness.csv',
    DATA_RESULTS / 'results_by_industry.csv',
    DATA_RESULTS / 'peer_coherence.csv',
    DATA_RESULTS / 'failure_analysis_worst1pct.csv',
    DATA_RESULTS / 'economic_significance_aggregate.csv',
    DATA_RESULTS / 'case_study_valuations.csv',
    DATA_RESULTS / 'ai_narrative_diffusion.csv',
    DATA_RESULTS / 'n10_robustness_checks.csv',
    FIGURES / 'n10_fusion_robustness.pdf',
    FIGURES / 'n10_all_models_by_year.pdf',
    FIGURES / 'n10_jaccard_overlap.pdf',
    FIGURES / 'n10_ff49_hit_rate.pdf',
    FIGURES / 'n10_case_study_ape.pdf',
    FIGURES / 'ai_narrative_diffusion.pdf',
]
# PURPOSE : Full evaluation — all models, all multiples, H1-H3, diagnostics, robustness checks
# RUNTIME : ~30 min (+ ~10 min for Sections 8.3–8.6)
# DEPENDS : peer files (N5-N9), multiples.parquet (N4), panel_clean.parquet (N1)

### 1 · Load Data

In [ ]:
# Load peer lists and multiples
peers_m0 = pd.read_parquet(PEERS_M0).dropna(subset=['focal_tic','peer_tic'])
peers_m1 = pd.read_parquet(PEERS_M1).dropna(subset=['focal_tic','peer_tic'])
peers_m2 = pd.read_parquet(PEERS_M2).dropna(subset=['focal_tic','peer_tic'])
peers_m3 = pd.read_parquet(PEERS_M3).dropna(subset=['focal_tic','peer_tic'])
peers_m3_rrf = pd.read_parquet(PEERS_M3_RRF).dropna(subset=['focal_tic','peer_tic'])

multiples = pd.read_parquet(MULTIPLES)
df_panel  = pd.read_parquet(
    PANEL_CLEAN,
    columns=['tic', 'fyear', 'ff49_num', 'ff49_abbr', 'industry', 'sic']
)

with open(ALPHA_OPTIMAL) as f:
    alpha_info = json.load(f)

print(f"M0  : {len(peers_m0):>8,} records | {peers_m0['focal_tic'].nunique():,} focal firms")
print(f"M1  : {len(peers_m1):>8,} records | {peers_m1['focal_tic'].nunique():,} focal firms")
print(f"M2  : {len(peers_m2):>8,} records | {peers_m2['focal_tic'].nunique():,} focal firms")
print(f"M3  : {len(peers_m3):>8,} records | {peers_m3['focal_tic'].nunique():,} focal firms")
print(f"M3r : {len(peers_m3_rrf):>8,} records (RRF)")
print(f"\nOptimal alpha: {alpha_info['best_alpha']}  "
      f"(val MdAPE={alpha_info['best_mdape_val']*100:.3f}%)")

In [ ]:
# Build evaluation sample from valid Gemini summaries
INVALID_FLAGS = {'ERROR', 'INSUFFICIENT_DATA', 'ERROR_EXTRACTING_TEXT'}

summary_tickers = set()
for yr in YEARS:
    path = SUMMARIES_FILES[yr]
    if not path.exists():
        continue
    df_s = pd.read_csv(path)
    valid = df_s[
        df_s['business_description'].notna() &
        ~df_s['business_description'].isin(INVALID_FLAGS) &
        (df_s['business_description'].str.len() > 50)
    ]
    for _, row in valid.iterrows():
        summary_tickers.add((row['tic'], int(row['fyear'])))

print(f"Full panel firm-years                : {len(multiples):,}")
print(f"Firm-years with valid Gemini summary : {len(summary_tickers):,}")
print(f"Evaluation sample                    : {len(summary_tickers):,} "
      f"({len(summary_tickers)/len(multiples)*100:.1f}% of panel)")

def restrict_to_eval_sample(peers_df, eval_set):
    mask = peers_df.apply(
        lambda r: (r['focal_tic'], int(r['focal_fyear'])) in eval_set, axis=1
    )
    return peers_df[mask].copy()

peers_m0     = restrict_to_eval_sample(peers_m0,     summary_tickers)
peers_m1     = restrict_to_eval_sample(peers_m1,     summary_tickers)
peers_m2     = restrict_to_eval_sample(peers_m2,     summary_tickers)
peers_m3     = restrict_to_eval_sample(peers_m3,     summary_tickers)
peers_m3_rrf = restrict_to_eval_sample(peers_m3_rrf, summary_tickers)

multiples = multiples[multiples.apply(
    lambda r: (r['tic'], int(r['fyear'])) in summary_tickers, axis=1
)].copy()

print("\nAfter restriction to evaluation sample:")
for name, df in [('M0', peers_m0), ('M1', peers_m1), ('M2', peers_m2),
                  ('M3', peers_m3), ('M3r', peers_m3_rrf)]:
    print(f"  {name:<5}: {len(df):>8,} records")
print(f"  Multiples : {len(multiples):>8,} firm-years")
print(f"  Unique firms: {multiples['tic'].nunique():,}")

### 2 · Evaluation Functions

In [ ]:
def compute_mdape(peers_df, multiples_df, multiple_col, k):
    if peers_df['rank'].max() > 1:
        peers_k = peers_df[peers_df['rank'] <= k].copy()
    else:
        peers_k = peers_df.copy()

    mult_lookup = (multiples_df[['tic', 'fyear', multiple_col]]
                   .dropna(subset=[multiple_col]))
    peers_k = peers_k.merge(
        mult_lookup.rename(columns={
            'tic': 'peer_tic', 'fyear': 'focal_fyear',
            multiple_col: 'peer_multiple'
        }),
        on=['peer_tic', 'focal_fyear'], how='inner'
    )
    predicted = (peers_k.groupby(['focal_tic', 'focal_fyear'])['peer_multiple']
                        .median().reset_index()
                        .rename(columns={'peer_multiple': 'predicted'}))
    predicted = predicted.merge(
        mult_lookup.rename(columns={
            'tic': 'focal_tic', 'fyear': 'focal_fyear',
            multiple_col: 'actual'
        }),
        on=['focal_tic', 'focal_fyear'], how='inner'
    )
    predicted['ape'] = ((predicted['actual'] - predicted['predicted']).abs() /
                        predicted['actual'].abs())
    return predicted[['focal_tic', 'focal_fyear', 'ape']].dropna()


def bootstrap_mdape(error_df, n_iter=BOOTSTRAP_ITERS, ci=CI_LEVEL):
    errors = error_df['ape'].values
    rng    = np.random.RandomState(RANDOM_SEED)
    boots  = [np.median(rng.choice(errors, len(errors), replace=True)) for _ in range(n_iter)]
    alpha  = (1 - ci) / 2
    return {
        'mdape': np.median(errors),
        'ci_lo': np.percentile(boots, alpha * 100),
        'ci_hi': np.percentile(boots, (1 - alpha) * 100),
        'mape' : np.mean(errors),
        'n'    : len(errors),
    }


def evaluate_model(peers_df, multiples_df, multiple_col, k, model_name):
    err    = compute_mdape(peers_df, multiples_df, multiple_col, k)
    result = bootstrap_mdape(err)
    result.update({'model': model_name, 'multiple': multiple_col, 'k': k})
    return result, err


print("Evaluation functions defined.")

### 3 · Main Results Grid

In [ ]:
MODELS = [
    ('M0_FF49',       peers_m0),
    ('M1_Financial',  peers_m1),
    ('M2_Text',       peers_m2),
    ('M3_Fusion',     peers_m3),
    ('M3_Fusion_RRF', peers_m3_rrf),
]

MULTIPLES_EVAL = [
    ('ln_v2s', 'ln(EV/Sales)'),
    ('ln_v2a', 'ln(EV/Assets)'),
    ('ln_m2b', 'ln(MktCap/SEQ)'),
]

results      = []
errors_store = {}

print("Running full evaluation grid...")
for model_name, peers_df in MODELS:
    for mult_col, mult_label in MULTIPLES_EVAL:
        for k in K_ROBUSTNESS:
            result, err = evaluate_model(peers_df, multiples, mult_col, k, model_name)
            results.append(result)
            errors_store[(model_name, mult_col, k)] = err
    print(f"  {model_name} done.")

results_df = pd.DataFrame(results)
print(f"\nTotal result rows: {len(results_df)}")

#### 3.1 · Main Results Table (k=10)

In [ ]:
print("=" * 90)
print(f"MAIN RESULTS TABLE — k={K_MAIN}, Bootstrapped {CI_LEVEL*100:.0f}% CI")
print("=" * 90)

for mult_col, mult_label in MULTIPLES_EVAL:
    print(f"\n  {mult_label}:")
    print(f"  {'Model':<20} {'MdAPE':>8} {'CI Lo':>8} {'CI Hi':>8} {'n':>7}")
    print(f"  {'-' * 55}")
    sub = results_df[(results_df['multiple'] == mult_col) &
                     (results_df['k'] == K_MAIN) &
                     (~results_df['model'].str.contains('RRF'))]
    for _, row in sub.iterrows():
        primary = " <- primary" if mult_col == 'ln_v2s' else ""
        print(f"  {row['model']:<20} "
              f"{row['mdape']*100:>7.2f}%  "
              f"{row['ci_lo']*100:>7.2f}%  "
              f"{row['ci_hi']*100:>7.2f}%  "
              f"{row['n']:>7,}{primary}")

In [ ]:
# Robustness: MAPE alongside MdAPE
# Reuses errors_store (per-firm-year APE) — no new pipeline run needed.
print("=" * 90)
print(f"MAPE vs MdAPE COMPARISON — k={K_MAIN}")
print("=" * 90)

mape_rows = []
for mult_col, mult_label in MULTIPLES_EVAL:
    print(f"\n  {mult_label}:")
    print(f"  {'Model':<20} {'MdAPE':>8} {'MAPE':>9} {'Δ (MAPE-MdAPE)':>17} "
          f"{'MdAPE rank':>11} {'MAPE rank':>10}")
    print(f"  {'-' * 80}")

    # Collect both metrics for every model at this multiple/k
    mult_results = []
    for model_name, _ in MODELS:
        if 'RRF' in model_name:
            continue
        err = errors_store[(model_name, mult_col, K_MAIN)]
        mdape = err['ape'].median()
        mape  = err['ape'].mean()
        mult_results.append({
            'model': model_name,
            'mdape': mdape,
            'mape':  mape,
            'multiple': mult_col,
            'k': K_MAIN,
            'n': len(err),
        })

    sub = pd.DataFrame(mult_results)
    sub['mdape_rank'] = sub['mdape'].rank().astype(int)
    sub['mape_rank']  = sub['mape'].rank().astype(int)
    sub['rank_shift'] = sub['mdape_rank'] != sub['mape_rank']

    for _, row in sub.iterrows():
        flag = " *" if row['rank_shift'] else ""
        print(f"  {row['model']:<20} "
              f"{row['mdape']*100:>7.2f}%  "
              f"{row['mape']*100:>8.2f}%  "
              f"{(row['mape']-row['mdape'])*100:>+15.2f}pp  "
              f"{row['mdape_rank']:>10}  "
              f"{row['mape_rank']:>9}{flag}")

    mape_rows.append(sub)

mape_df = pd.concat(mape_rows, ignore_index=True)

# ── Robustness verdict ────────────────────────────────────────────────────────
print("\n" + "=" * 90)
print("RANKING STABILITY")
print("=" * 90)
for mult_col, mult_label in MULTIPLES_EVAL:
    sub = mape_df[mape_df['multiple'] == mult_col]
    stable = (sub['mdape_rank'] == sub['mape_rank']).all()
    verdict = "PRESERVED" if stable else "DIVERGENT"
    print(f"  {mult_label:<25} → ranking {verdict}")

    # Best-on-MdAPE vs best-on-MAPE
    best_mdape = sub.loc[sub['mdape'].idxmin(), 'model']
    best_mape  = sub.loc[sub['mape'].idxmin(),  'model']
    if best_mdape != best_mape:
        print(f"    ! Best on MdAPE: {best_mdape}  |  Best on MAPE: {best_mape}")

# Save for §5.5 robustness table if rankings hold
mape_df.to_csv(DATA_RESULTS / "mape_robustness.csv", index=False)
print(f"\nSaved: {DATA_RESULTS / 'mape_robustness.csv'}")

#### 3.2 · Fusion Method Robustness — Weighted Rank vs RRF

In [ ]:
print("=" * 80)
print("FUSION METHOD ROBUSTNESS — Weighted Rank vs RRF")
print("=" * 80)
print(f"\n{'Multiple':<16} {'k':>4}  {'WR MdAPE':>10} {'RRF MdAPE':>11} {'Diff (pp)':>10}")
print("-" * 55)

for mult_col, mult_label in MULTIPLES_EVAL:
    for k in K_ROBUSTNESS:
        wr  = results_df[(results_df['model']=='M3_Fusion') &
                         (results_df['multiple']==mult_col) &
                         (results_df['k']==k)]['mdape'].values
        rrf = results_df[(results_df['model']=='M3_Fusion_RRF') &
                         (results_df['multiple']==mult_col) &
                         (results_df['k']==k)]['mdape'].values
        if len(wr) and len(rrf):
            diff = (rrf[0] - wr[0]) * 100
            print(f"  {mult_label:<14} {k:>4}  "
                  f"{wr[0]*100:>9.3f}%  {rrf[0]*100:>10.3f}%  "
                  f"{diff:>+9.3f}pp")
    print()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

for model, color, marker, ls, label in [
    ('M3_Fusion',     C3, 'o', '-',  f'Weighted Rank (α={alpha_info["best_alpha"]})'),
    ('M3_Fusion_RRF', C3, 's', '--', f'RRF (k={RRF_K_CONSTANT})'),
    ('M1_Financial',  C1, '^', ':',  'M1 Financial (best single)'),
]:
    sub = results_df[(results_df['model']==model) &
                     (results_df['multiple']=='ln_v2s')].sort_values('k')
    ax.plot(sub['k'], sub['mdape']*100, color=color, marker=marker,
            linewidth=2, markersize=6, linestyle=ls, label=label,
            alpha=0.7 if 'RRF' in model else 1.0)
    ax.fill_between(sub['k'], sub['ci_lo']*100, sub['ci_hi']*100,
                    alpha=0.08, color=color)

ax.set_xlabel('k (number of peers)')
ax.set_ylabel('MdAPE (%)')
ax.set_title('Fusion Method Robustness — WR vs RRF (ln(EV/Sales))', fontsize=10)
ax.set_xticks(K_ROBUSTNESS)
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES / 'n10_fusion_robustness.pdf', dpi=FIGURE_DPI)
plt.show()
print(f"Saved: {FIGURES / 'n10_fusion_robustness.pdf'}")

### 4 · Hypothesis Tests (H1, H2, H3)

One-sided Wilcoxon signed-rank test on matched firm-level APE differences.

| Hypothesis | Null | Alternative |
|---|---|---|
| H1 | M1 ≥ M0 | M1 < M0 |
| H2 | M2 ≥ M0 | M2 < M0 |
| H3 | M3 ≥ best(M1,M2) | M3 < best |

#### 4.1 · H1 — Financial kNN Beats FF49

In [ ]:
print("=" * 70)
print("H1: Financial kNN (M1) outperforms FF49 Baseline (M0)")
print("=" * 70)

for mult_col, mult_label in MULTIPLES_EVAL:
    e0 = errors_store[('M0_FF49',      mult_col, K_MAIN)]
    e1 = errors_store[('M1_Financial', mult_col, K_MAIN)]

    merged = e0.merge(e1, on=['focal_tic','focal_fyear'], suffixes=('_m0','_m1'))
    stat, pval = wilcoxon(merged['ape_m0'], merged['ape_m1'], alternative='greater')

    m0_mdape = results_df[(results_df['model']=='M0_FF49') &
                          (results_df['multiple']==mult_col) &
                          (results_df['k']==K_MAIN)]['mdape'].values[0]
    m1_mdape = results_df[(results_df['model']=='M1_Financial') &
                          (results_df['multiple']==mult_col) &
                          (results_df['k']==K_MAIN)]['mdape'].values[0]
    imp = (m0_mdape - m1_mdape) / m0_mdape * 100

    sig = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.10 else 'n.s.'
    print(f"\n  {mult_label}:")
    print(f"    M0={m0_mdape*100:.2f}%  M1={m1_mdape*100:.2f}%  "
          f"Δ={imp:+.1f}%  p={pval:.4f} {sig}")

#### 4.2 · H2 — Text kNN Beats FF49

In [ ]:
print("=" * 70)
print("H2: Text kNN (M2) outperforms FF49 Baseline (M0)")
print("=" * 70)

for mult_col, mult_label in MULTIPLES_EVAL:
    e0 = errors_store[('M0_FF49', mult_col, K_MAIN)]
    e2 = errors_store[('M2_Text', mult_col, K_MAIN)]
    merged = e0.merge(e2, on=['focal_tic','focal_fyear'], suffixes=('_m0','_m2'))
    stat, pval = wilcoxon(merged['ape_m0'], merged['ape_m2'], alternative='greater')

    m0_mdape = results_df[(results_df['model']=='M0_FF49') &
                          (results_df['multiple']==mult_col) &
                          (results_df['k']==K_MAIN)]['mdape'].values[0]
    m2_mdape = results_df[(results_df['model']=='M2_Text') &
                          (results_df['multiple']==mult_col) &
                          (results_df['k']==K_MAIN)]['mdape'].values[0]
    imp = (m0_mdape - m2_mdape) / m0_mdape * 100
    sig = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.10 else 'n.s.'
    print(f"\n  {mult_label}:")
    print(f"    M0={m0_mdape*100:.2f}%  M2={m2_mdape*100:.2f}%  "
          f"Δ={imp:+.1f}%  p={pval:.4f} {sig}")

#### 4.3 · H3 — Fusion Beats Best Single Modality

In [ ]:
print("=" * 70)
print("H3: Fusion (M3) outperforms best single modality")
print("=" * 70)

for mult_col, mult_label in MULTIPLES_EVAL:
    m1_mdape = results_df[(results_df['model']=='M1_Financial') &
                          (results_df['multiple']==mult_col) &
                          (results_df['k']==K_MAIN)]['mdape'].values[0]
    m2_mdape = results_df[(results_df['model']=='M2_Text') &
                          (results_df['multiple']==mult_col) &
                          (results_df['k']==K_MAIN)]['mdape'].values[0]
    m3_mdape = results_df[(results_df['model']=='M3_Fusion') &
                          (results_df['multiple']==mult_col) &
                          (results_df['k']==K_MAIN)]['mdape'].values[0]

    best_single      = min(m1_mdape, m2_mdape)
    best_single_name = 'M1' if m1_mdape < m2_mdape else 'M2'
    imp = (best_single - m3_mdape) / best_single * 100

    e_best = errors_store[('M1_Financial' if best_single_name=='M1' else 'M2_Text',
                            mult_col, K_MAIN)]
    e3     = errors_store[('M3_Fusion', mult_col, K_MAIN)]
    merged = e_best.merge(e3, on=['focal_tic','focal_fyear'],
                          suffixes=('_best','_m3'))
    stat, pval = wilcoxon(merged['ape_best'], merged['ape_m3'], alternative='greater')
    sig = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.10 else 'n.s.'

    print(f"\n  {mult_label}:")
    print(f"    Best single ({best_single_name})={best_single*100:.2f}%  "
          f"M3={m3_mdape*100:.2f}%  Δ={imp:+.1f}%  p={pval:.4f} {sig}")

print(f"\n  RRF robustness check (ln_v2s, k={K_MAIN}):")
m3_wr  = results_df[(results_df['model']=='M3_Fusion') &
                    (results_df['multiple']=='ln_v2s') &
                    (results_df['k']==K_MAIN)]['mdape'].values[0]
m3_rrf = results_df[(results_df['model']=='M3_Fusion_RRF') &
                    (results_df['multiple']=='ln_v2s') &
                    (results_df['k']==K_MAIN)]['mdape'].values[0]
print(f"    M3 (WR)  = {m3_wr*100:.3f}%")
print(f"    M3 (RRF) = {m3_rrf*100:.3f}%")
print(f"    Difference = {abs(m3_wr-m3_rrf)*100:.3f}pp")

### 5 · Annual Breakdown

#### 5.1 · Year-by-Year MdAPE

In [ ]:
MODELS_PLOT = [
    ('M0_FF49',      peers_m0, C['M0_FF49'],      'o', '-'),
    ('M1_Financial', peers_m1, C['M1_Financial'], 's', '--'),
    ('M2_Text',      peers_m2, C['M2_Text'],      '^', '-.'),
    ('M3_Fusion',    peers_m3, C['M3_Fusion'],    'D', ':'),
]

annual = []
for yr in YEARS:
    for model_name, peers_df, color, marker, ls in MODELS_PLOT:
        peers_yr = peers_df[peers_df['focal_fyear']==yr]
        mult_yr  = multiples[multiples['fyear']==yr]
        err = compute_mdape(peers_yr, mult_yr, 'ln_v2s', K_MAIN)
        if len(err) > 10:
            rng   = np.random.RandomState(RANDOM_SEED)
            boots = [np.median(rng.choice(err['ape'].values, len(err), replace=True))
                     for _ in range(500)]
            annual.append({
                'model': model_name, 'fyear': yr,
                'mdape': np.median(err['ape']),
                'ci_lo': np.percentile(boots, 2.5),
                'ci_hi': np.percentile(boots, 97.5),
            })

ann_df = pd.DataFrame(annual)

fig, ax = plt.subplots(figsize=(10, 5))
for model_name, peers_df, color, marker, ls in MODELS_PLOT:
    sub = ann_df[ann_df['model']==model_name].sort_values('fyear')
    ax.plot(sub['fyear'], sub['mdape']*100, color=color, marker=marker,
            linewidth=2, markersize=6, linestyle=ls,
            label=MODEL_LABELS[model_name])
    ax.fill_between(sub['fyear'], sub['ci_lo']*100, sub['ci_hi']*100,
                    alpha=0.10, color=color)

ax.axvline(2022, color='#888888', linewidth=0.8, linestyle=':')
ax.text(2022.05, ann_df['ci_hi'].max()*100*0.97,
        'Rate hike', fontsize=7.5, color='#888888', rotation=90, va='top')
ax.set_xlabel('Fiscal Year')
ax.set_ylabel('MdAPE (%)')
ax.set_title(f'All Models — MdAPE by Year (ln(EV/Sales), k={K_MAIN})', fontsize=10)
ax.set_xticks(YEARS)
ax.legend(loc='upper left')
plt.tight_layout()
plt.savefig(FIGURES / 'n10_all_models_by_year.pdf', dpi=FIGURE_DPI)
plt.show()
print(f"Saved: {FIGURES / 'n10_all_models_by_year.pdf'}")

In [ ]:
# Per-year MdAPE table with gap columns
print("Per-year MdAPE breakdown — ln(EV/Sales), k=10")
print("=" * 70)
print(f"{'Year':<6} {'M0':>8} {'M1':>8} {'M2':>8} {'M3':>8} "
      f"{'M1-M0':>8} {'M2-M1':>8} {'M3-M1':>8}")
print("-" * 70)

for year in YEARS:
    row = {}
    for model_name in ['M0_FF49','M1_Financial','M2_Text','M3_Fusion']:
        err = errors_store[(model_name, 'ln_v2s', K_MAIN)]
        err_yr = err[err['focal_fyear'] == year]
        row[model_name] = np.median(err_yr['ape']) * 100 if len(err_yr) > 0 else np.nan

    print(f"  {year:<4} "
          f"{row['M0_FF49']:>8.2f} "
          f"{row['M1_Financial']:>8.2f} "
          f"{row['M2_Text']:>8.2f} "
          f"{row['M3_Fusion']:>8.2f} "
          f"{row['M0_FF49']-row['M1_Financial']:>8.2f} "
          f"{row['M2_Text']-row['M1_Financial']:>8.2f} "
          f"{row['M1_Financial']-row['M3_Fusion']:>8.2f}")

#### 5.2 · Industry-Level Breakdown

In [ ]:
# For every FF49 industry with sufficient firm-years, compute MdAPE per model
# on the primary multiple (ln_v2s) at k=10. Highlights where each model wins
# and where text/fusion add value beyond the FF49 baseline.

MIN_FIRMYEARS_PER_INDUSTRY = 50   # exclude thin industries from per-industry table

# Lookup focal_tic + focal_fyear -> FF49 industry
focal_industry = (df_panel[['tic', 'fyear', 'ff49_num', 'ff49_abbr']]
                  .rename(columns={'tic': 'focal_tic', 'fyear': 'focal_fyear'})
                  .drop_duplicates(['focal_tic', 'focal_fyear']))

industry_rows = []
for model_name, _ in MODELS:
    if 'RRF' in model_name:
        continue
    err = errors_store[(model_name, 'ln_v2s', K_MAIN)].copy()
    err = err.merge(focal_industry, on=['focal_tic', 'focal_fyear'], how='inner')

    for (ff49_num, ff49_abbr), grp in err.groupby(['ff49_num', 'ff49_abbr']):
        if len(grp) < MIN_FIRMYEARS_PER_INDUSTRY:
            continue
        boot = bootstrap_mdape(grp[['ape']])
        industry_rows.append({
            'ff49_num' : int(ff49_num),
            'ff49_abbr': ff49_abbr,
            'model'    : model_name,
            'mdape'    : boot['mdape'],
            'ci_lo'    : boot['ci_lo'],
            'ci_hi'    : boot['ci_hi'],
            'n'        : boot['n'],
        })

industry_df = pd.DataFrame(industry_rows)

# Wide table: industry × model -> MdAPE
industry_wide = (industry_df
                 .pivot_table(index=['ff49_num', 'ff49_abbr'],
                              columns='model', values='mdape')
                 .reset_index())

# n column from M0 (largest, since M0 covers full eval sample by industry)
n_lookup = (industry_df[industry_df['model'] == 'M0_FF49']
            [['ff49_num', 'n']]
            .rename(columns={'n': 'firm_years'}))
industry_wide = industry_wide.merge(n_lookup, on='ff49_num', how='left')

# Improvements
industry_wide['delta_M1_M0'] = industry_wide['M0_FF49']      - industry_wide['M1_Financial']
industry_wide['delta_M2_M0'] = industry_wide['M0_FF49']      - industry_wide['M2_Text']
industry_wide['delta_M3_M0'] = industry_wide['M0_FF49']      - industry_wide['M3_Fusion']
industry_wide['delta_M3_M1'] = industry_wide['M1_Financial'] - industry_wide['M3_Fusion']

industry_wide = industry_wide.sort_values('firm_years', ascending=False).reset_index(drop=True)

print("=" * 110)
print(f"FF49 INDUSTRY BREAKDOWN — MdAPE on ln(EV/Sales), k={K_MAIN}, "
      f"min {MIN_FIRMYEARS_PER_INDUSTRY} firm-years")
print("=" * 110)
print(f"  {'#':>3} {'Industry':<28} {'n':>6} "
      f"{'M0':>7} {'M1':>7} {'M2':>7} {'M3':>7} "
      f"{'ΔM3-M0':>8} {'ΔM3-M1':>8}")
print(f"  {'-' * 105}")
for _, r in industry_wide.iterrows():
    print(f"  {int(r['ff49_num']):>3} {r['ff49_abbr']:<28} {int(r['firm_years']):>6,} "
          f"{r['M0_FF49']*100:>6.1f}% {r['M1_Financial']*100:>6.1f}% "
          f"{r['M2_Text']*100:>6.1f}% {r['M3_Fusion']*100:>6.1f}% "
          f"{r['delta_M3_M0']*100:>+7.1f}pp {r['delta_M3_M1']*100:>+7.1f}pp")

industry_wide.to_csv(DATA_RESULTS / 'results_by_industry.csv', index=False)
print(f"\nSaved: {DATA_RESULTS / 'results_by_industry.csv'}")

In [ ]:
# Figure: Top-20 industries — model performance & fusion gain over FF49
top_n = 20
top_ind = industry_wide.head(top_n).iloc[::-1].copy()    # reverse for barh
labels  = [f"{a} (n={int(n):,})"
           for a, n in zip(top_ind['ff49_abbr'], top_ind['firm_years'])]

fig, axes = plt.subplots(1, 2, figsize=(13, 7), sharey=True)

ax = axes[0]
y = np.arange(len(top_ind))
h = 0.20
ax.barh(y - 1.5*h, top_ind['M0_FF49']      * 100, h, color=C['M0_FF49'],      label='M0 FF49')
ax.barh(y - 0.5*h, top_ind['M1_Financial'] * 100, h, color=C['M1_Financial'], label='M1 Financial')
ax.barh(y + 0.5*h, top_ind['M2_Text']      * 100, h, color=C['M2_Text'],      label='M2 Text')
ax.barh(y + 1.5*h, top_ind['M3_Fusion']    * 100, h, color=C['M3_Fusion'],    label='M3 Fusion')
ax.set_yticks(y); ax.set_yticklabels(labels)
ax.set_xlabel('MdAPE (%)')
ax.set_title(f'(a) MdAPE by FF49 industry — top {top_n} by sample size')
ax.legend(loc='lower right', frameon=True)
ax.grid(axis='x', alpha=0.4)

ax = axes[1]
delta = top_ind['delta_M3_M0'] * 100
colors = [C['M3_Fusion'] if d > 0 else CGREY for d in delta]
ax.barh(y, delta, color=colors, edgecolor='#333', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Δ MdAPE: M0 − M3 (percentage points)')
ax.set_title('(b) Fusion gain vs FF49 baseline')
ax.grid(axis='x', alpha=0.4)
for yi, d in zip(y, delta):
    ax.text(d + (0.4 if d >= 0 else -0.4), yi,
            f'{d:+.1f}', va='center',
            ha='left' if d >= 0 else 'right', fontsize=8)

plt.tight_layout()
fig_path = FIGURES / f"n10_industry_breakdown.{FIGURE_FORMAT}"
plt.savefig(fig_path)
plt.show()
print(f"Saved: {fig_path}")

# Headline numbers for prose
won_by_m3 = (industry_wide['delta_M3_M0'] > 0).sum()
total     = len(industry_wide)
print(f"\nM3 beats M0 in {won_by_m3}/{total} industries "
      f"({won_by_m3/total*100:.0f}%)")
print(f"Largest M3 gain : {industry_wide['ff49_abbr'].iloc[industry_wide['delta_M3_M0'].argmax()]} "
      f"({industry_wide['delta_M3_M0'].max()*100:+.1f}pp)")
print(f"Worst M3 result : {industry_wide['ff49_abbr'].iloc[industry_wide['delta_M3_M0'].argmin()]} "
      f"({industry_wide['delta_M3_M0'].min()*100:+.1f}pp)")

#### 5.3 · Peer Coherence Diagnostics

The methodology chapter (§4.6.2) defines two peer-set quality diagnostics that
complement MdAPE: **peer-to-focal similarity** (how close, on average, are the
top-$k$ peers to the focal firm) and **peer-to-peer coherence** (how tightly
clustered the peers are among themselves). Both are computed in two geometries —
the 64-dimensional financial space used by M1 and the 768-dimensional FinBERT
text space used by M2 — so each model can be evaluated in the dimension it
optimises against and in the dimension it does not.

The results below report these four quantities (focal–peer × peer–peer ×
financial × text) at $k=10$ on the evaluation sample. The hypothesis is that
each model produces tight clusters in its own modality and looser ones in the
other; M3, by construction, should sit between M1 and M2 on both axes.

In [ ]:
# Peer coherence diagnostics — focal-peer & peer-peer similarity in both spaces
# Reports M0/M1/M2/M3 on a 2×2 grid: {focal-peer, peer-peer} × {financial, text}
from sklearn.preprocessing import normalize

# 1. Build feature matrices (one per geometry), one row per (tic, fyear)
print("Loading feature matrices...")
fin = pd.read_parquet(FINANCIALS_NORM)
with open(SELECTED_FEATURES_FILE) as f:
    feat_cols = json.load(f)['selected_features']

fin_keys = fin[['tic', 'fyear']].copy()
fin_X    = fin[feat_cols].fillna(0).values
fin_X    = normalize(fin_X, norm='l2', axis=1)        # cosine == dot post-L2
fin_idx  = {(t, int(y)): i for i, (t, y) in enumerate(zip(fin_keys['tic'],
                                                           fin_keys['fyear']))}

emb = pd.read_parquet(EMBEDDINGS)
# robust column detection — embeddings stored as dim_0..dim_767
emb_dim_cols = [c for c in emb.columns if c not in ('tic', 'fyear')]
emb_X    = emb[emb_dim_cols].values.astype(np.float32)
emb_X    = normalize(emb_X, norm='l2', axis=1)
emb_idx  = {(t, int(y)): i for i, (t, y) in enumerate(zip(emb['tic'],
                                                           emb['fyear']))}

print(f"  Financial : {fin_X.shape[0]:,} rows × {fin_X.shape[1]} dims")
print(f"  Text      : {emb_X.shape[0]:,} rows × {emb_X.shape[1]} dims")


def coherence_for_model(peers_df, k):
    """Mean focal-peer & peer-peer cosine in both spaces, per focal firm-year."""
    peers_k = (peers_df[peers_df['rank'] <= k]
               if peers_df['rank'].max() > 1 else peers_df).copy()

    rows = []
    grouped = peers_k.groupby(['focal_tic', 'focal_fyear'])
    for (focal_tic, focal_year), grp in grouped:
        focal_year = int(focal_year)
        peer_tics  = grp['peer_tic'].tolist()
        if len(peer_tics) < 2:
            continue

        for space, X, lookup in [('fin', fin_X, fin_idx),
                                  ('txt', emb_X, emb_idx)]:
            f_key = (focal_tic, focal_year)
            p_keys = [(p, focal_year) for p in peer_tics]
            f_idx_v = lookup.get(f_key)
            p_idx_v = [lookup[k_] for k_ in p_keys if k_ in lookup]
            if f_idx_v is None or len(p_idx_v) < 2:
                continue

            f_vec  = X[f_idx_v]
            p_vecs = X[p_idx_v]

            # focal-peer: mean cosine of focal vs each peer
            focal_peer = float((p_vecs @ f_vec).mean())
            # peer-peer: mean of upper-triangle of peer×peer cosine
            P = p_vecs @ p_vecs.T
            iu = np.triu_indices(len(p_vecs), k=1)
            peer_peer = float(P[iu].mean())

            rows.append({
                'focal_tic'  : focal_tic,
                'focal_fyear': focal_year,
                'space'      : space,
                'focal_peer' : focal_peer,
                'peer_peer'  : peer_peer,
            })
    return pd.DataFrame(rows)


coherence = {}
for model_name, peers_df in MODELS:
    if 'RRF' in model_name:
        continue
    print(f"  computing coherence for {model_name}...")
    coherence[model_name] = coherence_for_model(peers_df, K_MAIN)

# 2. Aggregate to model × space → mean focal-peer & peer-peer
agg_rows = []
for model_name, df in coherence.items():
    for space in ['fin', 'txt']:
        sub = df[df['space'] == space]
        agg_rows.append({
            'Model'     : model_name,
            'Space'     : 'Financial' if space == 'fin' else 'Text',
            'Focal-Peer': sub['focal_peer'].mean(),
            'Peer-Peer' : sub['peer_peer'].mean(),
            'n'         : len(sub),
        })
coh_df = pd.DataFrame(agg_rows)

print("\n" + "=" * 86)
print(f"PEER COHERENCE — k={K_MAIN}, eval sample, mean cosine similarity")
print("=" * 86)
for space_label in ['Financial', 'Text']:
    print(f"\n  {space_label} space:")
    print(f"    {'Model':<16} {'Focal-Peer':>12} {'Peer-Peer':>12} {'n':>8}")
    print(f"    {'-' * 52}")
    for _, r in coh_df[coh_df['Space'] == space_label].iterrows():
        print(f"    {r['Model']:<16} {r['Focal-Peer']:>12.3f} "
              f"{r['Peer-Peer']:>12.3f} {r['n']:>8,}")

coh_df.to_csv(DATA_RESULTS / 'peer_coherence.csv', index=False)
print(f"\nSaved: {DATA_RESULTS / 'peer_coherence.csv'}")

In [ ]:
# Figure: 2×2 grid of coherence — each model × {focal-peer, peer-peer} × space
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)

models_in_order = ['M0_FF49', 'M1_Financial', 'M2_Text', 'M3_Fusion']
x = np.arange(len(models_in_order))
w = 0.35

for ax, space_label in zip(axes, ['Financial', 'Text']):
    sub = coh_df[coh_df['Space'] == space_label].set_index('Model').loc[models_in_order]
    ax.bar(x - w/2, sub['Focal-Peer'], w,
           color=[C[m] for m in models_in_order], alpha=0.95,
           edgecolor='white', linewidth=0.6, label='Focal–Peer')
    ax.bar(x + w/2, sub['Peer-Peer'], w,
           color=[C[m] for m in models_in_order], alpha=0.55,
           edgecolor='white', linewidth=0.6, label='Peer–Peer', hatch='//')
    ax.set_xticks(x)
    ax.set_xticklabels([MODEL_LABELS[m].split(':')[0] for m in models_in_order],
                       rotation=0)
    ax.set_title(f'{space_label} space')
    ax.set_ylabel('Mean cosine similarity')
    ax.set_ylim(0, 1)
    ax.grid(axis='y', alpha=0.4)
    ax.legend(loc='lower right', frameon=True, fontsize=8)

plt.suptitle(f'Peer coherence by model and geometry (k={K_MAIN})',
             fontsize=11, y=1.02)
plt.tight_layout()
fig_path = FIGURES / f"n10_peer_coherence.{FIGURE_FORMAT}"
plt.savefig(fig_path)
plt.show()
print(f"Saved: {fig_path}")

#### 5.4 · Failure Analysis: Where Does Late Fusion Break Down?

Aggregate MdAPE statistics summarise central tendency but obscure the
right tail. Section §6.5 noted Amazon as a structural outlier whose
EV/Sales multiple no peer median can reasonably reach. To formalise this
intuition we examine the worst-1% of focal firm-years under M3 — the
cases where late fusion fails — and characterise them along four
dimensions: market-cap decile, R&D-disclosure regime, FF49 industry,
and 10-K summary word count.

Three failure modes are anticipated. First, **structural outliers**
(Amazon, Tesla, Netflix in growth phases) trade at multiples no peer
median can match regardless of peer-set quality. Second, **thin-text
firms** with summaries near the 50-character minimum supply M2 with
weak signal. Third, **size-boundary firms** sit just above the
\$50M-market-cap filter and have few size-comparable peers in the eval
sample. The table below reports, for each failure mode, its share among
the worst 1% of M3 firm-years and its share in the full eval sample —
a ratio greater than one indicates over-representation.

In [ ]:
# Failure analysis — worst 1% of focal firm-years under M3
# Characterise along: size decile, R&D regime, FF49, summary word count
WORST_PCT = 0.01

# 1. Per-firm-year APE under M3
m3_err = errors_store[('M3_Fusion', 'ln_v2s', K_MAIN)].copy()
m3_err = m3_err.rename(columns={'focal_tic': 'tic', 'focal_fyear': 'fyear'})

# 2. Attach characteristics from panel (size, xrd, ff49, sic)
char = pd.read_parquet(
    PANEL_CLEAN,
    columns=['tic', 'fyear', 'csho', 'prcc_f', 'xrd', 'sale',
             'ff49_num', 'ff49_abbr', 'sic']
).copy()
char['market_cap'] = char['csho'] * char['prcc_f']

# 3. Size decile per fyear
char['size_decile'] = (char.groupby('fyear')['market_cap']
                            .transform(lambda s: pd.qcut(s, 10,
                                                          labels=False,
                                                          duplicates='drop') + 1))

# 4. R&D regime
char['rd_regime'] = np.where(char['xrd'].fillna(0) > 0, 'R&D-Active', 'R&D-Zero')

# 5. Summary word count
summary_wc = []
for yr, path in SUMMARIES_FILES.items():
    if not path.exists():
        continue
    s = pd.read_csv(path)
    s = s[s['business_description'].notna() &
          ~s['business_description'].isin(['ERROR', 'INSUFFICIENT_DATA',
                                              'ERROR_EXTRACTING_TEXT'])]
    s['word_count'] = s['business_description'].str.split().str.len()
    s['fyear'] = int(yr)
    summary_wc.append(s[['tic', 'fyear', 'word_count']])
summary_wc = pd.concat(summary_wc, ignore_index=True)

# 6. Merge it all
m3_err = m3_err.merge(char, on=['tic', 'fyear'], how='left')
m3_err = m3_err.merge(summary_wc, on=['tic', 'fyear'], how='left')

# 7. Worst 1% by APE
threshold = m3_err['ape'].quantile(1 - WORST_PCT)
worst = m3_err[m3_err['ape'] >= threshold].copy()
n_worst = len(worst)
n_full  = len(m3_err)

print("=" * 90)
print(f"FAILURE ANALYSIS — worst {WORST_PCT*100:.0f}% of focal firm-years under M3")
print("=" * 90)
print(f"  APE threshold (top-{WORST_PCT*100:.0f}%): {threshold*100:.1f}%")
print(f"  n worst: {n_worst:,}   n full eval sample: {n_full:,}")

# ── Failure mode 1: structural outliers (top 3 deciles by size + extreme APE) ──
print(f"\n  Failure mode 1 — Size profile of worst firms:")
print(f"    {'Decile':<8} {'Worst share':>12} {'Full share':>12} {'Over-rep':>10}")
print(f"    {'-' * 48}")
for d in sorted(worst['size_decile'].dropna().unique().astype(int)):
    w_share = (worst['size_decile'] == d).sum() / n_worst
    f_share = (m3_err['size_decile'] == d).sum() / n_full
    over    = w_share / f_share if f_share > 0 else np.nan
    print(f"    {int(d):<8} {w_share*100:>11.1f}% {f_share*100:>11.1f}% "
          f"{over:>9.2f}x")

# ── Failure mode 2: R&D regime ─────────────────────────────────────────────
print(f"\n  Failure mode 2 — R&D regime of worst firms:")
for regime in ['R&D-Active', 'R&D-Zero']:
    w_share = (worst['rd_regime'] == regime).sum() / n_worst
    f_share = (m3_err['rd_regime'] == regime).sum() / n_full
    over    = w_share / f_share if f_share > 0 else np.nan
    print(f"    {regime:<14} worst={w_share*100:>5.1f}%  "
          f"full={f_share*100:>5.1f}%  over-rep={over:.2f}x")

# ── Failure mode 3: thin-text (summary < 200 words) ────────────────────────
thin_thresh = 200
worst_thin  = (worst['word_count'] < thin_thresh).sum()
full_thin   = (m3_err['word_count'] < thin_thresh).sum()
print(f"\n  Failure mode 3 — Thin-text (summary < {thin_thresh} words):")
print(f"    Worst-1%: {worst_thin/n_worst*100:>5.1f}%  ({worst_thin:,} firms)")
print(f"    Full    : {full_thin/n_full*100:>5.1f}%  ({full_thin:,} firms)")
print(f"    Over-representation: "
      f"{(worst_thin/n_worst)/(full_thin/n_full):.2f}x")

# ── Top FF49 industries among the worst ────────────────────────────────────
print(f"\n  Top FF49 industries in worst-1% (showing share ≥ 3%):")
ff49_worst = (worst['ff49_abbr'].value_counts(normalize=True) * 100).head(10)
ff49_full  = (m3_err['ff49_abbr'].value_counts(normalize=True) * 100)
for ind, w_pct in ff49_worst.items():
    if w_pct < 3.0:
        continue
    f_pct = ff49_full.get(ind, 0)
    over  = w_pct / f_pct if f_pct > 0 else np.nan
    print(f"    {ind:<28} worst={w_pct:>5.1f}%  full={f_pct:>5.1f}%  "
          f"over-rep={over:.2f}x")

# ── Top 20 named worst firms ───────────────────────────────────────────────
print(f"\n  Top 20 worst firm-years (by APE under M3):")
top20 = worst.nlargest(20, 'ape')[
    ['tic', 'fyear', 'ape', 'market_cap', 'rd_regime', 'word_count', 'ff49_abbr']
]
print(f"    {'Tic':<7} {'Year':>5} {'APE':>7} {'Mkt Cap $M':>11} "
      f"{'R&D':<11} {'Words':>6} {'Industry':<22}")
print(f"    {'-' * 80}")
for _, r in top20.iterrows():
    print(f"    {r['tic']:<7} {int(r['fyear']):>5} "
          f"{r['ape']*100:>6.0f}% {r['market_cap']:>11,.0f} "
          f"{r['rd_regime']:<11} {int(r['word_count']) if pd.notna(r['word_count']) else 0:>6} "
          f"{r['ff49_abbr']:<22}")

worst.to_csv(DATA_RESULTS / 'failure_analysis_worst1pct.csv', index=False)
print(f"\nSaved: {DATA_RESULTS / 'failure_analysis_worst1pct.csv'}")

### 6 · Economic Significance

In [ ]:
# For each focal firm-year, "warranted EV" = peer-median (EV/Sales) × focal Sales.
# Dollar valuation gap |Warranted EV − Actual EV| translates the MdAPE
# improvement into M&A / IPO / fairness-opinion-relevant magnitudes.

# Need the *raw* (un-logged) ev_sales multiple plus focal-firm sale, mkt cap, EV
# We rebuild from the panel because multiples.parquet stores log values only.
panel_full = pd.read_parquet(
    PANEL_CLEAN,
    columns=['tic', 'fyear', 'sale', 'at', 'csho', 'prcc_f',
             'dltt', 'dlc', 'che', 'seq', 'ff49_num', 'ff49_abbr', 'industry']
).copy()

panel_full['market_cap'] = panel_full['csho'] * panel_full['prcc_f']
panel_full['ev']         = panel_full['market_cap'] + panel_full['dltt'].fillna(0) \
                                                    + panel_full['dlc'].fillna(0) \
                                                    - panel_full['che'].fillna(0)
panel_full['ev_sales_raw'] = panel_full['ev'] / panel_full['sale']

# Winsorise raw multiple at 1/99 per fyear (consistent with N4)
panel_full['ev_sales_w'] = panel_full['ev_sales_raw'].copy()
for yr, sub in panel_full.groupby('fyear'):
    lo, hi = sub['ev_sales_raw'].quantile([0.01, 0.99])
    mask = panel_full['fyear'] == yr
    panel_full.loc[mask, 'ev_sales_w'] = panel_full.loc[mask, 'ev_sales_raw'].clip(lo, hi)

# Restrict to evaluation sample
panel_full = panel_full[panel_full.apply(
    lambda r: (r['tic'], int(r['fyear'])) in summary_tickers, axis=1
)].copy()

print(f"Eval-sample firm-years with usable EV: {panel_full['ev'].gt(0).sum():,}")


def warranted_ev(peers_df, panel_df, k):
    """Predict EV from peer-median EV/Sales × focal Sales."""
    peers_k = (peers_df[peers_df['rank'] <= k]
               if peers_df['rank'].max() > 1 else peers_df).copy()

    peer_mult = (panel_df[['tic', 'fyear', 'ev_sales_w']]
                 .rename(columns={'tic': 'peer_tic',
                                  'fyear': 'focal_fyear',
                                  'ev_sales_w': 'peer_mult'})
                 .dropna())

    peers_k = peers_k.merge(peer_mult, on=['peer_tic', 'focal_fyear'], how='inner')
    pred    = (peers_k.groupby(['focal_tic', 'focal_fyear'])['peer_mult']
                      .median().reset_index()
                      .rename(columns={'peer_mult': 'pred_mult'}))

    focal = panel_df[['tic', 'fyear', 'sale', 'ev', 'market_cap',
                      'ff49_abbr', 'industry']].rename(
        columns={'tic': 'focal_tic', 'fyear': 'focal_fyear'})
    pred  = pred.merge(focal, on=['focal_tic', 'focal_fyear'], how='inner')

    pred['warranted_ev']  = pred['pred_mult'] * pred['sale']
    pred['ev_gap_usd']    = pred['warranted_ev'] - pred['ev']
    pred['ev_gap_abs']    = pred['ev_gap_usd'].abs()
    pred['ev_gap_pct']    = pred['ev_gap_usd'] / pred['ev']
    return pred


# Compute warranted EV for each model (k = K_MAIN)
warranted = {}
for model_name, peers_df in MODELS:
    if 'RRF' in model_name:
        continue
    warranted[model_name] = warranted_ev(peers_df, panel_full, K_MAIN)
    print(f"  {model_name}: {len(warranted[model_name]):,} firm-years priced")

In [ ]:
agg_rows = []
for model_name in ['M0_FF49', 'M1_Financial', 'M2_Text', 'M3_Fusion']:
    df = warranted[model_name].dropna(subset=['ev_gap_usd', 'ev'])
    df = df[df['ev'] > 0]
    agg_rows.append({
        'Model'                 : model_name,
        'Firm-years priced'     : len(df),
        'Mean |gap| ($M)'       : df['ev_gap_abs'].mean(),
        'Median |gap| ($M)'     : df['ev_gap_abs'].median(),
        'Mean |gap| / EV (%)'   : df['ev_gap_abs'].div(df['ev']).mean()   * 100,
        'Median |gap| / EV (%)' : df['ev_gap_abs'].div(df['ev']).median() * 100,
        'Sum |gap| ($B)'        : df['ev_gap_abs'].sum() / 1e3,   # $M -> $B
    })

agg_df = pd.DataFrame(agg_rows).set_index('Model')
print("=" * 92)
print(f"PORTFOLIO-LEVEL DOLLAR MAGNITUDES — k={K_MAIN}, eval sample")
print("=" * 92)
print(agg_df.round(1).to_string())

m0_total = agg_df.loc['M0_FF49',      'Sum |gap| ($B)']
m1_total = agg_df.loc['M1_Financial', 'Sum |gap| ($B)']
m2_total = agg_df.loc['M2_Text',      'Sum |gap| ($B)']
m3_total = agg_df.loc['M3_Fusion',    'Sum |gap| ($B)']

print(f"\nTotal absolute valuation error:")
print(f"  M0 FF49      : ${m0_total:>10,.0f}B")
print(f"  M1 Financial : ${m1_total:>10,.0f}B   "
      f"(reduction vs M0: ${m0_total - m1_total:>7,.0f}B, "
      f"{(m0_total - m1_total)/m0_total*100:>5.1f}%)")
print(f"  M2 Text      : ${m2_total:>10,.0f}B   "
      f"(reduction vs M0: ${m0_total - m2_total:>7,.0f}B, "
      f"{(m0_total - m2_total)/m0_total*100:>5.1f}%)")
print(f"  M3 Fusion    : ${m3_total:>10,.0f}B   "
      f"(reduction vs M0: ${m0_total - m3_total:>7,.0f}B, "
      f"{(m0_total - m3_total)/m0_total*100:>5.1f}%)")

m0_med = agg_df.loc['M0_FF49',      'Median |gap| ($M)']
m1_med = agg_df.loc['M1_Financial', 'Median |gap| ($M)']
m2_med = agg_df.loc['M2_Text',      'Median |gap| ($M)']
m3_med = agg_df.loc['M3_Fusion',    'Median |gap| ($M)']

print(f"\nMedian per-firm-year |valuation gap|:")
print(f"  M0 FF49      : ${m0_med:>10,.0f}M")
print(f"  M1 Financial : ${m1_med:>10,.0f}M   "
      f"(reduction vs M0: ${m0_med - m1_med:>5,.0f}M, "
      f"{(m0_med - m1_med)/m0_med*100:>5.1f}%)")
print(f"  M2 Text      : ${m2_med:>10,.0f}M   "
      f"(reduction vs M0: ${m0_med - m2_med:>5,.0f}M, "
      f"{(m0_med - m2_med)/m0_med*100:>5.1f}%)")
print(f"  M3 Fusion    : ${m3_med:>10,.0f}M   "
      f"(reduction vs M0: ${m0_med - m3_med:>5,.0f}M, "
      f"{(m0_med - m3_med)/m0_med*100:>5.1f}%)")

agg_df.to_csv(DATA_RESULTS / 'economic_significance_aggregate.csv')

In [ ]:
# Figure: Distribution of |EV gap| / EV by model — supports Eaton et al. (2022)
# bridge in Discussion §6.X (economic significance)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel (a): |gap| / EV distribution, M0 vs M3
ax = axes[0]
for model_name, color, label in [
    ('M0_FF49',   C['M0_FF49'],   MODEL_LABELS['M0_FF49']),
    ('M3_Fusion', C['M3_Fusion'], MODEL_LABELS['M3_Fusion']),
]:
    df = warranted[model_name].dropna(subset=['ev_gap_pct'])
    df = df[df['ev'] > 0]
    pct_abs = df['ev_gap_pct'].abs() * 100
    pct_abs = pct_abs[pct_abs <= 200]
    ax.hist(pct_abs, bins=50, color=color, alpha=0.55, label=label,
            edgecolor='white', linewidth=0.4)
    ax.axvline(pct_abs.median(), color=color, linestyle='--',
               linewidth=1.4, alpha=0.9)
ax.set_xlabel('|Warranted EV − Actual EV| / Actual EV  (%)')
ax.set_ylabel('Firm-years')
ax.set_title('(a) Distribution of absolute EV mispricing')
ax.legend(loc='upper right', frameon=True)

# Panel (b): cumulative % of firms within X% mispricing
ax = axes[1]
for model_name, color, label in [
    ('M0_FF49',      C['M0_FF49'],      MODEL_LABELS['M0_FF49']),
    ('M1_Financial', C['M1_Financial'], MODEL_LABELS['M1_Financial']),
    ('M2_Text',      C['M2_Text'],      MODEL_LABELS['M2_Text']),
    ('M3_Fusion',    C['M3_Fusion'],    MODEL_LABELS['M3_Fusion']),
]:
    df = warranted[model_name].dropna(subset=['ev_gap_pct'])
    df = df[df['ev'] > 0]
    pct_abs = df['ev_gap_pct'].abs() * 100
    pct_abs = pct_abs.sort_values()
    cum = np.arange(1, len(pct_abs) + 1) / len(pct_abs) * 100
    ax.plot(pct_abs.values, cum, color=color, label=label, linewidth=1.6)
ax.set_xlim(0, 150)
ax.set_xlabel('|Warranted EV − Actual EV| / Actual EV  (%)')
ax.set_ylabel('Cumulative % of firm-years')
ax.set_title('(b) CDF of mispricing — closer to top-left is better')
ax.legend(loc='lower right', frameon=True)
ax.axhline(50, color='black', linewidth=0.4, alpha=0.5, linestyle=':')

plt.tight_layout()
fig_path = FIGURES / f"n10_economic_significance.{FIGURE_FORMAT}"
plt.savefig(fig_path)
plt.show()
print(f"Saved: {fig_path}")

### 7 · Peer Set Complementarity — Jaccard Overlap

Jaccard(A,B) = |A ∩ B| / |A ∪ B| over top-k peer lists.
Low Jaccard = the models find different peers, i.e. they capture complementary signals.

In [ ]:
# Build lookup dicts
m1_lookup = (peers_m1[peers_m1['rank'] <= K_MAIN]
             .groupby(['focal_tic', 'focal_fyear'])['peer_tic']
             .apply(set).to_dict())
m2_lookup = (peers_m2[peers_m2['rank'] <= K_MAIN]
             .groupby(['focal_tic', 'focal_fyear'])['peer_tic']
             .apply(set).to_dict())

# Compute Jaccard for every focal firm-year present in BOTH peer lists
all_keys = set(m1_lookup.keys()) & set(m2_lookup.keys())
print(f"Computing Jaccard for {len(all_keys):,} focal firm-years...")

records = []
for (tic, yr) in all_keys:
    m1_set = m1_lookup[(tic, yr)]
    m2_set = m2_lookup[(tic, yr)]
    union  = m1_set | m2_set
    if not union:
        continue
    records.append({
        'focal_tic'  : tic,
        'focal_fyear': yr,
        'jaccard'    : len(m1_set & m2_set) / len(union),
    })

jac_df = pd.DataFrame(records)

print(f"\n{'=' * 60}")
print(f"JACCARD OVERLAP M1 vs M2  (k={K_MAIN})")
print(f"{'=' * 60}")
print(f"  n firm-years : {len(jac_df):,}")
print(f"  Mean         : {jac_df['jaccard'].mean():.4f}")
print(f"  Median       : {jac_df['jaccard'].median():.4f}")
print(f"  Std          : {jac_df['jaccard'].std():.4f}")
print(f"  % with J=0   : {(jac_df['jaccard'] == 0).mean() * 100:.1f}%")
print(f"  % with J>0.2 : {(jac_df['jaccard'] > 0.2).mean() * 100:.1f}%")
print(f"  % with J>0.5 : {(jac_df['jaccard'] > 0.5).mean() * 100:.1f}%")

In [ ]:
# Figure — Jaccard histogram (full eval sample)
fig, ax = plt.subplots(figsize=(7, 4.5))

ax.hist(jac_df['jaccard'], bins=50, color='#2166ac', alpha=0.8,
        edgecolor='none', label='All firm-years')
ax.axvline(jac_df['jaccard'].median(), color='#d6604d', linewidth=1.8,
           linestyle='--',
           label=f"Median = {jac_df['jaccard'].median():.3f}")
ax.set_xlabel(f'Jaccard Similarity (k={K_MAIN})')
ax.set_ylabel('Number of focal firm-years')
ax.set_title(f'M1 vs M2 Peer Set Overlap  (k={K_MAIN})', fontsize=10)
ax.legend(fontsize=9)

pct_zero = (jac_df['jaccard'] == 0).mean() * 100
ax.text(0.62, 0.82, f'{pct_zero:.0f}% of firms\nhave J = 0',
        transform=ax.transAxes, fontsize=9, color='#333333',
        bbox=dict(boxstyle='round,pad=0.3',
                  facecolor='#f0f0f0', alpha=0.8))

plt.tight_layout()
plt.savefig(FIGURES / 'n10_jaccard_overlap.pdf', dpi=FIGURE_DPI,
            bbox_inches='tight')
plt.show()
print(f"Saved: {FIGURES / 'n10_jaccard_overlap.pdf'}")

### 8 · FF49 Same-Industry Hit Rate

In [ ]:
df_industry = df_panel[['tic', 'fyear', 'ff49_num', 'industry']].drop_duplicates()

def compute_ff49_hit_rate(peers_df, df_industry, k):
    """Fraction of top-k peers in the same FF49 industry as focal firm."""
    if peers_df['rank'].max() > 1:
        peers_k = peers_df[peers_df['rank'] <= k].copy()
    else:
        peers_k = peers_df.copy()

    peers_k = peers_k.merge(
        df_industry.rename(columns={
            'tic': 'focal_tic', 'fyear': 'focal_fyear',
            'ff49_num': 'focal_ff49', 'industry': 'focal_industry'
        }),
        on=['focal_tic', 'focal_fyear'], how='left'
    )
    peers_k = peers_k.merge(
        df_industry.rename(columns={
            'tic': 'peer_tic', 'fyear': 'focal_fyear',
            'ff49_num': 'peer_ff49', 'industry': 'peer_industry'
        }),
        on=['peer_tic', 'focal_fyear'], how='left'
    )
    peers_k['same_ff49'] = peers_k['focal_ff49'] == peers_k['peer_ff49']
    per_firm = (peers_k.groupby(['focal_tic', 'focal_fyear'])['same_ff49']
                        .mean().reset_index()
                        .rename(columns={'same_ff49': 'hit_rate'}))
    return per_firm['hit_rate']


MODELS_EVAL = [
    ('M0_FF49',      peers_m0),
    ('M1_Financial', peers_m1),
    ('M2_Text',      peers_m2),
    ('M3_Fusion',    peers_m3),
]

hit_rate_store = {}

print(f"\n{'Model':<20} {'k=5':>8} {'k=10':>8} {'k=15':>8} {'k=20':>8}")
print("-" * 52)

for model_name, peers_df in MODELS_EVAL:
    row_str = f"  {model_name:<18}"
    for k in K_ROBUSTNESS:
        rates = compute_ff49_hit_rate(peers_df, df_industry, k)
        mean_rate = rates.mean() * 100
        hit_rate_store[(model_name, k)] = rates
        row_str += f"  {mean_rate:>6.1f}%"
    print(row_str)

# Per-year breakdown for k=10
print(f"\n\nPer-year breakdown (k={K_MAIN}):")
print(f"{'Model':<20} " + "  ".join(f"{yr:>8}" for yr in YEARS))
print("-" * 65)

for model_name, peers_df in MODELS_EVAL:
    row_str = f"  {model_name:<18}"
    for yr in YEARS:
        peers_yr = peers_df[peers_df['focal_fyear'] == yr]
        if len(peers_yr) == 0:
            row_str += f"  {'—':>7}"
            continue
        rates = compute_ff49_hit_rate(peers_yr, df_industry, K_MAIN)
        row_str += f"  {rates.mean()*100:>6.1f}%"
    print(row_str)

In [ ]:
# Each model's peer list is fixed. MdAPE changes with (i) the k used to
# truncate the peer list and (ii) the multiple we price with. The table
# isolates these two sources of variation. M0 is k-invariant by design
# (equal-weight same-industry peers — no ranking to truncate).
# =============================================================================

def _mdape(peers_df, mult_col, k):
    return compute_mdape(peers_df, multiples, mult_col, k)['ape'].median() * 100

for mult_col, mult_label in MULTIPLES_EVAL:
    n = multiples[mult_col].notna().sum()
    print(f"\n{'='*62}")
    print(f" {mult_label}   |   n = {n:,} firm-years")
    print('='*62)
    print(f"  {'Model':<16}" + "".join(f"{f'k={k}':>10}" for k in K_ROBUSTNESS))
    print("  " + "-" * 46)
    for model_name, peers_df in MODELS_EVAL:
        row = f"  {model_name:<16}"
        for k in K_ROBUSTNESS:
            row += f"{_mdape(peers_df, mult_col, k):>9.2f}%"
        print(row)

# ── Compact cross-target summary at k=K_MAIN ────────────────────────────────
print(f"\n\n{'='*72}")
print(f" MdAPE by target  |  k = {K_MAIN}")
print('='*72)
print(f"  {'Model':<16}" + "".join(f"{lbl:>18}" for _, lbl in MULTIPLES_EVAL))
print("  " + "-" * 70)
for model_name, peers_df in MODELS_EVAL:
    row = f"  {model_name:<16}"
    for mult_col, _ in MULTIPLES_EVAL:
        row += f"{_mdape(peers_df, mult_col, K_MAIN):>17.2f}%"
    print(row)

In [ ]:
# Figure — hit rate distribution by model
fig, axes = plt.subplots(1, len(MODELS_EVAL), figsize=(14, 4), sharey=True)
colors_list = [C0, C1, C2, C3]

for ax, (model_name, peers_df), color in zip(axes, MODELS_EVAL, colors_list):
    rates = hit_rate_store[(model_name, K_MAIN)] * 100
    ax.hist(rates, bins=30, color=color, alpha=0.85, edgecolor='none')
    ax.axvline(rates.mean(), color='black', linewidth=1.5,
               linestyle='--', label=f'Mean {rates.mean():.1f}%')
    ax.set_title(MODEL_LABELS[model_name], fontsize=9)
    ax.set_xlabel('% peers in same FF49 industry')
    ax.legend(fontsize=8)

axes[0].set_ylabel('Number of focal firms')
plt.suptitle(f'FF49 Same-Industry Hit Rate Distribution (k={K_MAIN})',
             fontsize=10, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES / 'n10_ff49_hit_rate.pdf', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print(f"Saved: {FIGURES / 'n10_ff49_hit_rate.pdf'}")

### 9 · Robustness Checks

#### 9.1 · December FYE Subsample

In [ ]:
df_dates = pd.read_parquet(PANEL_CLEAN, columns=['tic','fyear','datadate'])
df_dates['datadate'] = pd.to_datetime(df_dates['datadate'])
df_dates['fiscal_month'] = df_dates['datadate'].dt.month
dec_set = set(zip(
    df_dates[df_dates['fiscal_month']==12]['tic'],
    df_dates[df_dates['fiscal_month']==12]['fyear']
))

print(f"December fiscal year-end firm-years: {len(dec_set):,}")
print(f"\nRobustness — December FYE only (ln_v2s, k=10):")
print(f"  {'Model':<20} {'Full sample':>12} {'Dec only':>12}")
print("  " + "-"*46)

for model_name, peers_df, *_ in MODELS_PLOT:
    r_full = results_df[(results_df['model']==model_name) &
                        (results_df['multiple']=='ln_v2s') &
                        (results_df['k']==K_MAIN)]['mdape'].values[0]
    peers_dec = peers_df[peers_df.apply(
        lambda r: (r['focal_tic'], r['focal_fyear']) in dec_set, axis=1
    )]
    err_dec = compute_mdape(peers_dec, multiples, 'ln_v2s', K_MAIN)
    r_dec   = np.median(err_dec['ape'])
    print(f"  {model_name:<20} {r_full*100:>11.2f}%  {r_dec*100:>11.2f}%")

#### 9.2 · Validation vs Test Year Split

In [ ]:
results_split = []

for period_name, years in [
    ('Validation (2020-2022)', VALIDATION_YEARS),
    ('Test (2023-2024)',       TEST_YEARS),
    ('Full (2020-2024)',       YEARS),
]:
    print(f"  {period_name}:")
    print(f"  {'Model':<20} {'MdAPE':>8} {'n':>7}")
    print(f"  {'-'*38}")
    for model_name, peers_df in MODELS_EVAL:
        peers_period = peers_df[peers_df['focal_fyear'].isin(years)]
        mult_period  = multiples[multiples['fyear'].isin(years)]
        err = compute_mdape(peers_period, mult_period, 'ln_v2s', K_MAIN)
        if len(err) > 0:
            mdape = np.median(err['ape']) * 100
            n     = len(err)
            results_split.append({
                'period': period_name, 'model': model_name,
                'mdape': mdape, 'n': n,
            })
            print(f"  {model_name:<20} {mdape:>7.2f}%  {n:>7,}")
    print()

# H3 check: does M3 improvement over M1 hold on test years?
val_m1  = next(r['mdape'] for r in results_split
               if r['model']=='M1_Financial' and 'Validation' in r['period'])
val_m3  = next(r['mdape'] for r in results_split
               if r['model']=='M3_Fusion' and 'Validation' in r['period'])
test_m1 = next(r['mdape'] for r in results_split
               if r['model']=='M1_Financial' and 'Test' in r['period'])
test_m3 = next(r['mdape'] for r in results_split
               if r['model']=='M3_Fusion' and 'Test' in r['period'])

val_delta  = (val_m1  - val_m3)  / val_m1  * 100
test_delta = (test_m1 - test_m3) / test_m1 * 100

print("M3 improvement over M1 (H3):")
print(f"  Validation years: Δ = +{val_delta:.1f}%")
print(f"  Test years      : Δ = +{test_delta:.1f}%")
print("\n(If test Δ ≈ validation Δ, α generalises well.)")

#### 9.3 · Constrained Peer Selection

In [ ]:
# additional inputs for Sections 8.3–8.6
from sklearn.preprocessing import normalize as sk_normalize

finn_rob = pd.read_parquet(FINANCIALS_NORM)
emb_rob  = pd.read_parquet(EMBEDDINGS)
emb_cols_rob = [c for c in emb_rob.columns if c not in ('tic', 'fyear')]

with open(SELECTED_FEATURES_FILE) as f:
    _raw = json.load(f)
selected_features_rob = (
    _raw['selected_features'] if isinstance(_raw, dict) and 'selected_features' in _raw
    else list(_raw) if isinstance(_raw, list) else list(_raw.keys())
)

alpha_rob   = alpha_info['best_alpha']
panel_rob   = pd.read_parquet(PANEL_CLEAN,
    columns=['tic', 'fyear', 'ff49_num', 'industry', 'market_cap'])
p_focal_rob = panel_rob.rename(columns={'tic': 'focal_tic', 'fyear': 'focal_fyear'})
peer_ln_r   = multiples[['tic', 'fyear', 'ln_v2s']].rename(
    columns={'tic': 'peer_tic', 'fyear': 'focal_fyear', 'ln_v2s': 'peer_ln_v2s'})
focal_ln_r  = multiples[['tic', 'fyear', 'ln_v2s']].rename(
    columns={'tic': 'focal_tic', 'fyear': 'focal_fyear', 'ln_v2s': 'actual_ln'})
eval_idx_rob = summary_tickers

all_rob_results = []

def eval_ape_rob(peers_df, label, restrict_focals=None):
    merged = peers_df.merge(peer_ln_r, on=['peer_tic', 'focal_fyear'], how='left')
    pred = (merged.groupby(['focal_tic', 'focal_fyear'])['peer_ln_v2s']
            .median().reset_index().rename(columns={'peer_ln_v2s': f'pred_{label}'}))
    pred = pred.merge(focal_ln_r, on=['focal_tic', 'focal_fyear'], how='left')
    if restrict_focals is not None:
        key = list(zip(pred['focal_tic'], pred['focal_fyear']))
        pred = pred[[k in restrict_focals for k in key]]
    pred = pred.dropna(subset=['actual_ln', f'pred_{label}'])
    pred = pred[pred['actual_ln'] != 0]
    pred[f'ape_{label}'] = (
        (pred['actual_ln'] - pred[f'pred_{label}']).abs() / pred['actual_ln'].abs()
    )
    return pred

def build_m3_from_pair(m1_df, m2_df, alpha, k_out=K_MAIN, k_cand=20):
    m1 = m1_df[m1_df['rank'] <= k_cand][['focal_tic','focal_fyear','peer_tic','rank']].rename(
        columns={'rank': 'r_fin'})
    m2 = m2_df[m2_df['rank'] <= k_cand][['focal_tic','focal_fyear','peer_tic','rank']].rename(
        columns={'rank': 'r_text'})
    fused = m1.merge(m2, on=['focal_tic','focal_fyear','peer_tic'], how='outer')
    fused['score'] = (
        alpha * np.where(fused['r_text'].notna(), 1.0 / fused['r_text'], 0.0) +
        (1 - alpha) * np.where(fused['r_fin'].notna(), 1.0 / fused['r_fin'], 0.0)
    )
    fused = fused.sort_values(['focal_tic','focal_fyear','score'], ascending=[True,True,False])
    fused['rank'] = fused.groupby(['focal_tic','focal_fyear']).cumcount() + 1
    fused = fused[fused['rank'] <= k_out].copy()
    fused['similarity_score'] = fused['score']
    return fused[['focal_tic','focal_fyear','peer_tic','rank','similarity_score']]

print('Robustness helpers loaded.')
print(f'  finn: {len(finn_rob):,} rows, {len(selected_features_rob)} selected features')
print(f'  embeddings: {len(emb_rob):,} rows, {len(emb_cols_rob)} dims')
print(f'  alpha = {alpha_rob}')

In [ ]:
eval_df_rob = pd.DataFrame(list(eval_idx_rob), columns=['focal_tic', 'focal_fyear'])
eval_df_rob = eval_df_rob.merge(
    p_focal_rob[['focal_tic', 'focal_fyear', 'ff49_num']],
    on=['focal_tic', 'focal_fyear'], how='left'
)
ind_counts = (eval_df_rob.groupby(['ff49_num', 'focal_fyear'])
              .size().rename('ind_n').reset_index())
eval_with_ind = eval_df_rob.merge(ind_counts, on=['ff49_num', 'focal_fyear'], how='left')
eligible_focals_rob = set(zip(
    eval_with_ind.loc[eval_with_ind['ind_n'] >= K_MAIN + 1, 'focal_tic'],
    eval_with_ind.loc[eval_with_ind['ind_n'] >= K_MAIN + 1, 'focal_fyear']
))
print(f"Eval sample: {len(eval_idx_rob):,} focals")
print(f"Eligible (ind_n >= {K_MAIN+1}): {len(eligible_focals_rob):,}")
print(f"Dropped (thin industry): {len(eval_idx_rob) - len(eligible_focals_rob):,}")

In [ ]:
def build_constrained_peers_vectorised(feature_df, feature_cols, k, eligible_focals):
    out_chunks = []
    eligible_by_year = {}
    for ft, fy in eligible_focals:
        eligible_by_year.setdefault(fy, set()).add(ft)

    for (fyear, ff49), grp in feature_df.groupby(['focal_fyear', 'ff49_num']):
        if len(grp) < k + 1:
            continue
        tickers = grp['focal_tic'].to_numpy()
        X = grp[feature_cols].to_numpy(dtype=float)
        norms = np.linalg.norm(X, axis=1, keepdims=True)
        Xn = X / np.where(norms > 0, norms, 1.0)
        sim = Xn @ Xn.T
        np.fill_diagonal(sim, -np.inf)
        top_idx = np.argpartition(-sim, k, axis=1)[:, :k]
        row_idx = np.arange(len(tickers))[:, None]
        top_sims = sim[row_idx, top_idx]
        order = np.argsort(-top_sims, axis=1)
        top_idx_s = np.take_along_axis(top_idx, order, axis=1)
        top_sim_s = np.take_along_axis(top_sims, order, axis=1)
        eligible_yr = eligible_by_year.get(int(fyear), set())
        keep_rows = np.array([t in eligible_yr for t in tickers])
        if not keep_rows.any():
            continue
        focals = tickers[keep_rows]
        peers  = tickers[top_idx_s[keep_rows]]
        sims_k = top_sim_s[keep_rows]
        out_chunks.append(pd.DataFrame({
            'focal_tic':        np.repeat(focals, k),
            'focal_fyear':      np.int64(fyear),
            'peer_tic':         peers.reshape(-1),
            'rank':             np.tile(np.arange(1, k + 1), len(focals)),
            'similarity_score': sims_k.reshape(-1),
        }))
    return (pd.concat(out_chunks, ignore_index=True) if out_chunks
            else pd.DataFrame(columns=['focal_tic','focal_fyear','peer_tic','rank','similarity_score']))

ff49_map_rob  = p_focal_rob[['focal_tic', 'focal_fyear', 'ff49_num']]
finn_eval_rob = finn_rob.rename(columns={'tic': 'focal_tic', 'fyear': 'focal_fyear'}).copy()
finn_eval_rob = finn_eval_rob.drop(
    columns=[c for c in finn_eval_rob.columns if c.lower() in ('ff49_num', 'ff49')], errors='ignore'
).merge(ff49_map_rob, on=['focal_tic', 'focal_fyear'], how='left').dropna(subset=['ff49_num'])
finn_eval_rob = finn_eval_rob[
    finn_eval_rob.apply(lambda r: (r['focal_tic'], r['focal_fyear']) in eval_idx_rob, axis=1)]

print(f"finn_eval: {len(finn_eval_rob):,} rows")
print("M1 constrained: building within-industry top-K_MAIN ...")
peers_m1_c = build_constrained_peers_vectorised(
    finn_eval_rob, selected_features_rob, K_MAIN, eligible_focals_rob)
print(f"  rows: {len(peers_m1_c):,}  focals: {peers_m1_c['focal_tic'].nunique():,}")

emb_eval_rob = emb_rob.rename(columns={'tic': 'focal_tic', 'fyear': 'focal_fyear'}).copy()
emb_eval_rob = emb_eval_rob.merge(ff49_map_rob, on=['focal_tic', 'focal_fyear'], how='left').dropna(subset=['ff49_num'])
emb_eval_rob = emb_eval_rob[
    emb_eval_rob.apply(lambda r: (r['focal_tic'], r['focal_fyear']) in eval_idx_rob, axis=1)]

print("M2 constrained: building within-industry top-K_MAIN ...")
peers_m2_c = build_constrained_peers_vectorised(
    emb_eval_rob, emb_cols_rob, K_MAIN, eligible_focals_rob)
print(f"  rows: {len(peers_m2_c):,}  focals: {peers_m2_c['focal_tic'].nunique():,}")

print(f"M3 constrained: alpha={alpha_rob}, building k=20 candidates ...")
peers_m1_c_k20 = build_constrained_peers_vectorised(
    finn_eval_rob, selected_features_rob, 20, eligible_focals_rob)
peers_m2_c_k20 = build_constrained_peers_vectorised(
    emb_eval_rob, emb_cols_rob, 20, eligible_focals_rob)
peers_m3_c = build_m3_from_pair(peers_m1_c_k20, peers_m2_c_k20, alpha_rob)
print(f"  rows: {len(peers_m3_c):,}  focals: {peers_m3_c['focal_tic'].nunique():,}")

In [ ]:
r0  = eval_ape_rob(peers_m0,   'm0',   restrict_focals=eligible_focals_rob)
r1  = eval_ape_rob(peers_m1,   'm1',   restrict_focals=eligible_focals_rob)
r2  = eval_ape_rob(peers_m2,   'm2',   restrict_focals=eligible_focals_rob)
r3  = eval_ape_rob(peers_m3,   'm3',   restrict_focals=eligible_focals_rob)
r1c = eval_ape_rob(peers_m1_c, 'm1c',  restrict_focals=eligible_focals_rob)
r2c = eval_ape_rob(peers_m2_c, 'm2c',  restrict_focals=eligible_focals_rob)
r3c = eval_ape_rob(peers_m3_c, 'm3c',  restrict_focals=eligible_focals_rob)

print("== 8.3 — Within-industry MdAPE (ln_v2s), eligible focals ==")
print(f"{'Model':<34}{'MdAPE (%)':>12}{'n':>10}")
print("-" * 56)
for name, df, col in [
    ('M0 FF49 baseline (unchanged)',    r0,  'ape_m0'),
    ('M1 original (cross-industry)',    r1,  'ape_m1'),
    ('M1 constrained (within-ind.)',    r1c, 'ape_m1c'),
    ('M2 original (cross-industry)',    r2,  'ape_m2'),
    ('M2 constrained (within-ind.)',    r2c, 'ape_m2c'),
    ('M3 original (cross-industry)',    r3,  'ape_m3'),
    ('M3 constrained (within-ind.)',    r3c, 'ape_m3c'),
]:
    md_ = df[col].median() * 100 if len(df) else float('nan')
    print(f"{name:<34}{md_:>11.2f}%{len(df):>10,}")
    all_rob_results.append({'section': '8.3', 'variant': name, 'mdape_pct': md_, 'n': len(df)})

m0_v      = r0['ape_m0'].median() * 100
m1_orig_v = r1['ape_m1'].median() * 100
m1_con_v  = r1c['ape_m1c'].median() * 100
share = (m0_v - m1_con_v) / (m0_v - m1_orig_v) * 100 if (m0_v - m1_orig_v) != 0 else float('nan')
print(f"
Verdict:")
print(f"  M0 MdAPE: {m0_v:.2f}%  |  M1 gain over M0: {m0_v-m1_orig_v:+.2f}pp")
print(f"  M1 constrained gain over M0: {m0_v-m1_con_v:+.2f}pp")
print(f"  Share of M1 gain retained within industry: {share:.1f}%")

#### 9.4 · M0 Variants at k=10

In [ ]:
eligible_panel_rob = p_focal_rob[
    p_focal_rob.apply(lambda r: (r['focal_tic'], r['focal_fyear']) in eligible_focals_rob, axis=1)
]
industry_members_rob = (
    eligible_panel_rob.groupby(['ff49_num', 'focal_fyear'])['focal_tic'].apply(list).to_dict()
)
mkt_lookup_rob = p_focal_rob.set_index(['focal_tic', 'focal_fyear'])['market_cap'].to_dict()
focal_to_candidates_rob = {}
for ft, fy in eligible_focals_rob:
    row = eligible_panel_rob[
        (eligible_panel_rob['focal_tic'] == ft) & (eligible_panel_rob['focal_fyear'] == fy)]
    if row.empty:
        continue
    ff49  = row['ff49_num'].iloc[0]
    cands = [t for t in industry_members_rob.get((ff49, fy), []) if t != ft]
    if cands:
        focal_to_candidates_rob[(ft, fy)] = (cands, row['market_cap'].iloc[0])

print(f"Candidate pools: {len(focal_to_candidates_rob):,} eligible focals")

BOOTSTRAP_DRAWS = 50

def random_draw_peers(seed):
    rng = np.random.default_rng(seed)
    rows = []
    for (ft, fy), (cands, _) in focal_to_candidates_rob.items():
        k = min(K_MAIN, len(cands))
        picked = rng.choice(cands, size=k, replace=False)
        for r, peer in enumerate(picked, start=1):
            rows.append((ft, fy, peer, r, 1.0))
    return pd.DataFrame(rows, columns=['focal_tic','focal_fyear','peer_tic','rank','similarity_score'])

print(f"Variant A — random k={K_MAIN} averaged over {BOOTSTRAP_DRAWS} seeds ...")
mdapes_rand = []
for b in range(BOOTSTRAP_DRAWS):
    pdf = random_draw_peers(seed=b)
    r   = eval_ape_rob(pdf, 'm0_rand', restrict_focals=eligible_focals_rob)
    mdapes_rand.append(r['ape_m0_rand'].median() * 100)

mdape_rand_mean = float(np.mean(mdapes_rand))
mdape_rand_std  = float(np.std(mdapes_rand))
print(f"  MdAPE mean: {mdape_rand_mean:.2f}% (sd={mdape_rand_std:.3f}pp)")

print("Variant B — closest k by market cap within industry ...")
rows_b = []
for (ft, fy), (cands, focal_mcap) in focal_to_candidates_rob.items():
    if pd.isna(focal_mcap) or focal_mcap <= 0:
        continue
    log_focal  = np.log(focal_mcap)
    cand_dists = []
    for t in cands:
        mcap = mkt_lookup_rob.get((t, fy), float('nan'))
        if pd.isna(mcap) or mcap <= 0:
            continue
        cand_dists.append((t, abs(np.log(mcap) - log_focal)))
    cand_dists.sort(key=lambda x: x[1])
    for rnk, (peer, _) in enumerate(cand_dists[:K_MAIN], start=1):
        rows_b.append((ft, fy, peer, rnk, 1.0))

pdf_mcap   = pd.DataFrame(rows_b, columns=['focal_tic','focal_fyear','peer_tic','rank','similarity_score'])
r_mcap     = eval_ape_rob(pdf_mcap, 'm0_mcap', restrict_focals=eligible_focals_rob)
mdape_mcap = r_mcap['ape_m0_mcap'].median() * 100
m0_el      = r0['ape_m0'].median() * 100

print(f"
== 8.4 — M0 at k={K_MAIN} variants vs M0 full-industry and M1 ==")
print(f"  M0 full-industry:                {m0_el:.2f}%")
print(f"  M0 k={K_MAIN} random (B={BOOTSTRAP_DRAWS} mean):  {mdape_rand_mean:.2f}%  (sd={mdape_rand_std:.2f}pp)")
print(f"  M0 k={K_MAIN} closest-by-mktcap: {mdape_mcap:.2f}%")
print(f"  M1 original:                     {m1_orig_v:.2f}%")

shrink_gain = m0_el - mdape_rand_mean
total_gain  = m0_el - m1_orig_v
print(f"
Verdict:")
print(f"  List-size reduction alone (M0->random k): {shrink_gain:+.2f}pp")
if abs(total_gain) > 0.1:
    print(f"  Share of M1 gain from list-size reduction: {shrink_gain/total_gain*100:.1f}%")

for lbl, val in [
    ('M0 full-industry',           m0_el),
    ('M0 k=10 random (B=50 mean)', mdape_rand_mean),
    ('M0 k=10 closest-by-mktcap', mdape_mcap),
]:
    all_rob_results.append({'section': '8.4', 'variant': lbl, 'mdape_pct': val,
                             'n': len(eligible_focals_rob)})

#### 9.5 · Alpha* Stability Across k

In [ ]:
peers_m1_k20_rob = peers_m1[peers_m1['rank'] <= 20].copy()
peers_m2_k20_rob = peers_m2[peers_m2['rank'] <= 20].copy()

val_err_frame = errors_store[('M3_Fusion', 'ln_v2s', K_MAIN)]
val_focals_rob = set(zip(
    val_err_frame.loc[val_err_frame['focal_fyear'].isin(VALIDATION_YEARS), 'focal_tic'],
    val_err_frame.loc[val_err_frame['focal_fyear'].isin(VALIDATION_YEARS), 'focal_fyear'],
))
print(f"Validation focals: {len(val_focals_rob):,}")

k_grid_rob  = [5, 10, 15, 20]
rob3_records = []
for k_test in k_grid_rob:
    for a in ALPHA_GRID:
        peers_tmp = build_m3_from_pair(peers_m1_k20_rob, peers_m2_k20_rob, a,
                                       k_out=k_test, k_cand=20)
        r_tmp = eval_ape_rob(peers_tmp, 'm3_tmp', restrict_focals=val_focals_rob)
        md_ = r_tmp['ape_m3_tmp'].median() * 100 if len(r_tmp) else float('nan')
        rob3_records.append({'k': k_test, 'alpha': a, 'val_mdape_pct': md_, 'n': len(r_tmp)})

alpha_by_k_rob = pd.DataFrame(rob3_records)
pivot = alpha_by_k_rob.pivot_table(index='alpha', columns='k', values='val_mdape_pct')
print("
Validation MdAPE by (k, alpha):")
print(pivot.round(2).to_string())
print(f"
{'k':>4}{'alpha*':>10}{'val MdAPE':>14}")
opt_alphas = []
for k_test in k_grid_rob:
    sub  = alpha_by_k_rob[alpha_by_k_rob['k'] == k_test]
    best = sub.loc[sub['val_mdape_pct'].idxmin()]
    opt_alphas.append(float(best['alpha']))
    print(f"{k_test:>4}{best['alpha']:>10.2f}{best['val_mdape_pct']:>13.2f}%")
    all_rob_results.append({'section': '8.5',
                             'variant': f'k={k_test} alpha*={best["alpha"]:.1f}',
                             'mdape_pct': best['val_mdape_pct'], 'n': int(best['n'])})

if len(set(opt_alphas)) == 1:
    print(f"
Verdict: alpha* stable at {opt_alphas[0]} across all k.")
else:
    drift = max(opt_alphas) - min(opt_alphas)
    print(f"
Verdict: alpha* ranges {min(opt_alphas)}–{max(opt_alphas)} (drift {drift:.1f}).")

#### 9.6 · Financial Sector Exclusion

In [ ]:
EXCLUDE_FF49 = {44, 45, 47, 48, 31}

def drop_excluded_peers(peers_df, exclude_set):
    lookup = p_focal_rob[['focal_tic', 'focal_fyear', 'ff49_num']].rename(
        columns={'focal_tic': 'peer_tic', 'ff49_num': 'peer_ff49'})
    m = peers_df.merge(lookup, on=['peer_tic', 'focal_fyear'], how='left')
    m = m[~m['peer_ff49'].isin(exclude_set)]
    m = m.sort_values(['focal_tic', 'focal_fyear', 'rank'])
    m['rank'] = m.groupby(['focal_tic', 'focal_fyear']).cumcount() + 1
    return m[['focal_tic', 'focal_fyear', 'peer_tic', 'rank', 'similarity_score']]

peers_m1_excl     = drop_excluded_peers(peers_m1_k20_rob, EXCLUDE_FF49)
peers_m1_excl_k10 = peers_m1_excl[peers_m1_excl['rank'] <= K_MAIN].copy()
peers_m2_excl_k20 = drop_excluded_peers(peers_m2_k20_rob, EXCLUDE_FF49)
peers_m1_excl_k20 = drop_excluded_peers(peers_m1_k20_rob, EXCLUDE_FF49)
peers_m3_excl     = build_m3_from_pair(peers_m1_excl_k20, peers_m2_excl_k20, alpha_rob)

r1_excl   = eval_ape_rob(peers_m1_excl_k10, 'm1_excl')
r3_excl   = eval_ape_rob(peers_m3_excl,     'm3_excl')
r1_base   = eval_ape_rob(peers_m1,          'm1_base')
r3_base   = eval_ape_rob(peers_m3,          'm3_base')

md_m1_base = r1_base['ape_m1_base'].median() * 100
md_m1_excl = r1_excl['ape_m1_excl'].median() * 100
md_m3_base = r3_base['ape_m3_base'].median() * 100
md_m3_excl = r3_excl['ape_m3_excl'].median() * 100

print("== 8.6 — Financial/Utilities Exclusion from Candidate Pool ==")
print(f"  Excluded FF49 industries: {sorted(EXCLUDE_FF49)} (Banking, Insurance, Trading, Other Fin., Utilities)")
print(f"{'':32}{'Baseline':>10}{'Excluded':>10}{'Delta':>10}")
print(f"  {'M1 MdAPE':30}{md_m1_base:>9.2f}%{md_m1_excl:>9.2f}%{md_m1_excl-md_m1_base:>+9.2f}pp")
print(f"  {'M3 MdAPE':30}{md_m3_base:>9.2f}%{md_m3_excl:>9.2f}%{md_m3_excl-md_m3_base:>+9.2f}pp")
print(f"  n: {len(r1_base):,}")

non_fin_focals = (
    set(zip(
        p_focal_rob.loc[~p_focal_rob['ff49_num'].isin(EXCLUDE_FF49), 'focal_tic'],
        p_focal_rob.loc[~p_focal_rob['ff49_num'].isin(EXCLUDE_FF49), 'focal_fyear'],
    )) & eval_idx_rob
)
r1_base_nf = eval_ape_rob(peers_m1,          'm1_base_nf', restrict_focals=non_fin_focals)
r1_excl_nf = eval_ape_rob(peers_m1_excl_k10, 'm1_excl_nf', restrict_focals=non_fin_focals)
r3_base_nf = eval_ape_rob(peers_m3,          'm3_base_nf', restrict_focals=non_fin_focals)
r3_excl_nf = eval_ape_rob(peers_m3_excl,     'm3_excl_nf', restrict_focals=non_fin_focals)

print(f"
Non-financial focals only (n = {len(non_fin_focals):,}):")
print(f"  {'M1 MdAPE':30}"
      f"{r1_base_nf['ape_m1_base_nf'].median()*100:>9.2f}%"
      f"{r1_excl_nf['ape_m1_excl_nf'].median()*100:>9.2f}%"
      f"{(r1_excl_nf['ape_m1_excl_nf'].median()-r1_base_nf['ape_m1_base_nf'].median())*100:>+9.2f}pp")
print(f"  {'M3 MdAPE':30}"
      f"{r3_base_nf['ape_m3_base_nf'].median()*100:>9.2f}%"
      f"{r3_excl_nf['ape_m3_excl_nf'].median()*100:>9.2f}%"
      f"{(r3_excl_nf['ape_m3_excl_nf'].median()-r3_base_nf['ape_m3_base_nf'].median())*100:>+9.2f}pp")
print("
Verdict: |Δ| < 1pp for non-financial focals → retention is empirically defended.")

for lbl, val, n in [
    ('M1 base (all focals)',      md_m1_base, len(r1_base)),
    ('M1 excl (all focals)',      md_m1_excl, len(r1_excl)),
    ('M3 base (all focals)',      md_m3_base, len(r3_base)),
    ('M3 excl (all focals)',      md_m3_excl, len(r3_excl)),
    ('M1 base (non-fin focals)',  r1_base_nf['ape_m1_base_nf'].median()*100, len(r1_base_nf)),
    ('M1 excl (non-fin focals)',  r1_excl_nf['ape_m1_excl_nf'].median()*100, len(r1_excl_nf)),
]:
    all_rob_results.append({'section': '8.6', 'variant': lbl, 'mdape_pct': val, 'n': n})

In [ ]:
rob_df = pd.DataFrame(all_rob_results)
out_path = DATA_RESULTS / 'n10_robustness_checks.csv'
rob_df.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
print(f"Rows : {len(rob_df)}")
print(rob_df.to_string(index=False))

### 10 · Case Studies

In [ ]:
# Case study setup + peer-list figure (2024, role-balanced shortlist)

# Lookups (built once, reused throughout §10)
name_lookup = (pd.read_parquet(PANEL_CLEAN, columns=['tic', 'conm'])
               .drop_duplicates('tic').set_index('tic')['conm'].to_dict())
industry_lookup = (df_panel[['tic', 'fyear', 'ff49_num', 'industry']]
                   .set_index(['tic', 'fyear']))

def get_name(tic, maxlen=28):
    n = name_lookup.get(tic, tic)
    return n[:maxlen]

def get_ff49(tic, yr):
    try:    return industry_lookup.loc[(tic, yr), 'ff49_num']
    except KeyError: return None

FOCAL_FIRMS = [
    ('GILD', 2024, 'Pharmaceutical Products',    'Consensus-reinforcement'),
    ('TGT',  2024, 'Retail',                  'Baseline-survives'),
    ('NVDA', 2024, 'Semiconductors',          'Boundary case (scope)'),
    ('PFE',  2024, 'Pharmaceutical Products', 'Post-restructuring drift'),
    ('AMZN', 2024, 'Retail Stores',           'Boundary case (scope)'),
    ('GS',   2024, 'Trading',                    'Financial-dominant'),
]

ALL_PEERS = {'M0_FF49': peers_m0, 'M1_Financial': peers_m1,
             'M2_Text': peers_m2, 'M3_Fusion':    peers_m3}

# Sanity check
eval_focal = set(zip(peers_m3['focal_tic'], peers_m3['focal_fyear']))
for tic, yr, _, role in FOCAL_FIRMS:
    status = 'OK' if (tic, yr) in eval_focal else 'MISSING'
    print(f"  {tic:<6} FY{yr}  {role:<26}  {status}")

# ── Peer-list figure ────────────────────────────────────────────────────────
K_DISPLAY = 10
COLORS    = {'agree':   '#f5c842', 'ff49':    '#a8d8a8',
             'default': '#f0f0f0', 'm0':      '#d0e4f7',
             'header':  '#2c3e50'}

def peer_color(peer_tic, focal_ff49, m1_set, m2_set, model, yr):
    if model == 'M0_FF49': return COLORS['m0']
    if peer_tic in m1_set and peer_tic in m2_set: return COLORS['agree']
    if get_ff49(peer_tic, yr) == focal_ff49:      return COLORS['ff49']
    return COLORS['default']

def get_top_peers(tic, yr, peers_df, k=K_DISPLAY):
    return (peers_df[(peers_df['focal_tic'] == tic) &
                     (peers_df['focal_fyear'] == yr) &
                     (peers_df['rank'] <= k)]
            .sort_values('rank')['peer_tic'].tolist())

for focal_tic, yr, industry, role in FOCAL_FIRMS:
    focal_ff49 = get_ff49(focal_tic, yr)
    peer_lists = {m: get_top_peers(focal_tic, yr, df) for m, df in ALL_PEERS.items()}
    m1_set, m2_set = set(peer_lists['M1_Financial']), set(peer_lists['M2_Text'])

    fig, ax = plt.subplots(figsize=(13, 6.8))
    ax.axis('off')

    col_labels = ['Rank', 'M0: FF49 Baseline', 'M1: Financial kNN',
                  'M2: Text kNN (FinBERT)', 'M3: Late Fusion']
    col_widths  = [0.05, 0.24, 0.24, 0.24, 0.24]
    x_pos       = [sum(col_widths[:i]) for i in range(len(col_widths))]
    row_h, y0   = 0.14, 0.85

    # Header row
    for label, xp, w in zip(col_labels, x_pos, col_widths):
        ax.add_patch(plt.Rectangle((xp, y0), w, row_h, transform=ax.transAxes,
                                    color=COLORS['header'], clip_on=False))
        ax.text(xp + w/2, y0 + row_h/2, label, transform=ax.transAxes,
                ha='center', va='center', fontsize=8.5,
                color='white', fontweight='bold')

    # Peer rows
    models = list(ALL_PEERS.keys())
    for r_idx in range(K_DISPLAY):
        y = y0 - (r_idx + 1) * row_h
        # Rank cell
        ax.add_patch(plt.Rectangle((x_pos[0], y), col_widths[0], row_h,
                                    transform=ax.transAxes, color='#e8e8e8',
                                    clip_on=False, lw=0.5, ec='white'))
        ax.text(x_pos[0] + col_widths[0]/2, y + row_h/2, str(r_idx + 1),
                transform=ax.transAxes, ha='center', va='center',
                fontsize=8, color='#333333')
        # Model cells
        for c_idx, model in enumerate(models):
            xp, w = x_pos[c_idx + 1], col_widths[c_idx + 1]
            peers = peer_lists[model]
            if r_idx < len(peers):
                pt = peers[r_idx]
                color = peer_color(pt, focal_ff49, m1_set, m2_set, model, yr)
                star  = '  *' if (get_ff49(pt, yr) == focal_ff49 and
                                   model != 'M0_FF49') else ''
                label = f"{pt}  {get_name(pt)}{star}"
            else:
                color, label = '#ffffff', '—'
            ax.add_patch(plt.Rectangle((xp, y), w, row_h, transform=ax.transAxes,
                                        color=color, clip_on=False,
                                        lw=0.4, ec='white'))
            ax.text(xp + 0.008, y + row_h/2, label, transform=ax.transAxes,
                    ha='left', va='center', fontsize=7.5, color='#1a1a1a')

    # Legend
    legend_y = y0 - (K_DISPLAY + 1.2) * row_h
    legend_items = [
        (COLORS['agree'],   'Peer selected by both M1 and M2'),
        (COLORS['ff49'],    'Peer in same FF49 industry as focal firm  (*)'),
        (COLORS['default'], 'Unique to model, crosses FF49 boundary'),
        (COLORS['m0'],      'M0 peer (FF49 industry membership)'),
    ]
    for i, (color, lbl) in enumerate(legend_items):
        xp = 0.01 + i * 0.25
        ax.add_patch(plt.Rectangle((xp, legend_y), 0.018, row_h * 0.7,
                                    transform=ax.transAxes, color=color,
                                    clip_on=False, lw=0.4, ec='#aaaaaa'))
        ax.text(xp + 0.022, legend_y + row_h * 0.35, lbl,
                transform=ax.transAxes, ha='left', va='center',
                fontsize=7, color='#333333')

    fig.suptitle(f"Case Study — {role}: {focal_tic} ({get_name(focal_tic)})  |  "
                 f"{industry}  |  FY{yr}",
                 fontsize=11, fontweight='bold', y=1.01)
    fig.tight_layout()
    plt.savefig(FIGURES / f"n10_case_{focal_tic.lower()}.pdf",
                dpi=FIGURE_DPI, bbox_inches='tight')
    plt.show()

    # Jaccard summary
    inter, union = m1_set & m2_set, m1_set | m2_set
    j = len(inter) / len(union) if union else 0
    shared = ', '.join(sorted(inter)) if inter else 'none'
    print(f"  {focal_tic}: Jaccard(M1,M2) = {j:.3f}  |  Shared: {shared}\n")

#### 10.1 · Implied Valuations and APE

In [ ]:
# Implied valuations + APE bar chart — now with $ EV columns

K_VAL = 10

# Pull sales and EV directly from the parquet (df_panel may not have these columns)
case_years = [yr for _, yr, _, _ in FOCAL_FIRMS]
focal_financials = (pd.read_parquet(PANEL_CLEAN, columns=['tic', 'fyear', 'sale', 'ev'])
                    .pipe(lambda d: d[d['fyear'].isin(case_years)])
                    .rename(columns={'tic': 'focal_tic', 'fyear': 'focal_fyear'}))

with open(ALPHA_OPTIMAL) as f:
    best_alpha = json.load(f)['best_alpha']  # 0.3

def implied_valuation(focal_tic, yr, peers_df, mult_df,
                       financials_df, col='ln_v2s', k=K_VAL, model_name=None):
    peers_k = (peers_df[(peers_df['focal_tic'] == focal_tic) &
                         (peers_df['focal_fyear'] == yr) &
                         (peers_df['rank'] <= k)]
               .merge(mult_df.loc[mult_df['fyear'] == yr,
                                   ['tic', col]].rename(
                          columns={'tic': 'peer_tic', col: 'peer_mult'}),
                      on='peer_tic', how='left'))
    implied_ln = peers_k['peer_mult'].median()
    actual_row = mult_df[(mult_df['fyear'] == yr) &
                          (mult_df['tic']   == focal_tic)][col]
    fin_row = financials_df[(financials_df['focal_tic']   == focal_tic) &
                              (financials_df['focal_fyear'] == yr)]
    if actual_row.empty or pd.isna(implied_ln) or fin_row.empty:
        return None

    # ── cosine: for M3 blend M1+M2 cosines; others use stored score ──────
    if model_name == 'M3_Fusion':
        m1_scores = (peers_m1[(peers_m1['focal_tic'] == focal_tic) &
                               (peers_m1['focal_fyear'] == yr) &
                               (peers_m1['peer_tic'].isin(peers_k['peer_tic']))]
                     .set_index('peer_tic')['similarity_score'])
        m2_scores = (peers_m2[(peers_m2['focal_tic'] == focal_tic) &
                               (peers_m2['focal_fyear'] == yr) &
                               (peers_m2['peer_tic'].isin(peers_k['peer_tic']))]
                     .set_index('peer_tic')['similarity_score'])
        mean_cosine = ((1 - best_alpha) * m1_scores.mean() +
                       best_alpha       * m2_scores.mean())
    else:
        mean_cosine = float(peers_k['similarity_score'].mean())

    actual_mult  = float(np.exp(actual_row.values[0]))
    implied_mult = float(np.exp(implied_ln))
    sale         = float(fin_row['sale'].iloc[0])
    actual_ev    = float(fin_row['ev'].iloc[0])
    implied_ev   = implied_mult * sale

    return {
        'actual_mult':   actual_mult,
        'implied_mult':  implied_mult,
        'sale_$M':       sale,
        'actual_ev_$M':  actual_ev,
        'implied_ev_$M': implied_ev,
        'ev_gap_$M':     implied_ev - actual_ev,
        'ev_gap_pct':    (implied_ev - actual_ev) / abs(actual_ev) * 100,
        'ape':           abs(actual_mult - implied_mult) / abs(actual_mult) * 100,
        'n_peers':       int(peers_k['peer_mult'].notna().sum()),
        'mean_cosine':   float(mean_cosine),
    }

# Rebuild records passing model_name through
records = []
for focal_tic, yr, industry, role in FOCAL_FIRMS:
    for model_name, peers_df in ALL_PEERS.items():
        res = implied_valuation(focal_tic, yr, peers_df, multiples,
                                 focal_financials, model_name=model_name)
        if res is None: continue
        records.append({'focal_tic': focal_tic, 'industry': industry,
                        'role': role, 'model': model_name, **res})

val_df = pd.DataFrame(records)

# ── Print formatted summary per firm (now with $ EV columns) ───────────────
print("=" * 105)
print(f"  IMPLIED VALUATIONS — Multiple and $ EV  |  k={K_VAL}  |  FY2024")
print("=" * 105)
for focal_tic, yr, industry, role in FOCAL_FIRMS:
    sub = val_df[val_df['focal_tic'] == focal_tic]
    if sub.empty: continue
    actual_mult = sub['actual_mult'].iloc[0]
    actual_ev   = sub['actual_ev_$M'].iloc[0]
    sale        = sub['sale_$M'].iloc[0]
    print(f"\n  {focal_tic}  ({role})  —  {industry}")
    print(f"  Sales = ${sale:,.0f}M  |  Actual EV = ${actual_ev:,.0f}M  |  "
          f"Actual EV/Sales = {actual_mult:.2f}x")
    print(f"  {'Model':<14} {'Implied EV/S':>12} {'APE':>7}  "
          f"{'Implied EV $M':>14} {'EV gap $M':>13} {'EV gap %':>9} {'Mean cosine':>12}")  # ← +header
    print(f"  {'-' * 93}")
    for _, r in sub.iterrows():
        print(f"  {r['model']:<14} "
              f"{r['implied_mult']:>10.2f}x  "
              f"{r['ape']:>5.1f}%  "
              f"{r['implied_ev_$M']:>13,.0f}  "
              f"{r['ev_gap_$M']:>+12,.0f}  "
              f"{r['ev_gap_pct']:>+8.1f}%  "
              f"{r['mean_cosine']:>11.4f}")  # ← added

val_df.to_csv(DATA_RESULTS / 'case_study_valuations.csv', index=False)
print(f"\nSaved: {DATA_RESULTS / 'case_study_valuations.csv'}")

# ── Side-by-side compact $ EV summary table (one row per firm) ──────────────
print(f"\n{'=' * 105}")
print(f"  COMPACT $ EV TABLE  —  Implied EV by Model ($ millions)")
print(f"{'=' * 105}")
print(f"  {'Tic':<6} {'Role':<26} {'Actual EV':>12}  "
      f"{'M0 EV':>11} {'M1 EV':>11} {'M2 EV':>11} {'M3 EV':>11}")
print(f"  {'-' * 100}")
for focal_tic, yr, industry, role in FOCAL_FIRMS:
    sub = val_df[val_df['focal_tic'] == focal_tic]
    if sub.empty: continue
    actual_ev = sub['actual_ev_$M'].iloc[0]
    evs = {m: sub.loc[sub['model'] == m, 'implied_ev_$M'].values[0]
            if not sub[sub['model'] == m].empty else np.nan
            for m in ['M0_FF49', 'M1_Financial', 'M2_Text', 'M3_Fusion']}
    print(f"  {focal_tic:<6} {role:<26} {actual_ev:>11,.0f}  "
          f"{evs['M0_FF49']:>10,.0f}  {evs['M1_Financial']:>10,.0f}  "
          f"{evs['M2_Text']:>10,.0f}  {evs['M3_Fusion']:>10,.0f}")

# ── APE bar chart (unchanged from previous version) ────────────────────────
MODEL_ORDER = ['M0_FF49', 'M1_Financial', 'M2_Text', 'M3_Fusion']
MODEL_SHORT = {'M0_FF49':      'M0\nFF49',
               'M1_Financial': 'M1\nFinancial',
               'M2_Text':      'M2\nText',
               'M3_Fusion':    'M3\nFusion'}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
axes[5].axis('off')

for ax, (focal_tic, yr, industry, role) in zip(axes, FOCAL_FIRMS):
    sub = val_df[val_df['focal_tic'] == focal_tic]
    if sub.empty:
        ax.axis('off'); continue
    actual = sub['actual_mult'].iloc[0]

    apes = [sub.loc[sub['model'] == m, 'ape'].values[0]
             if not sub[sub['model'] == m].empty else np.nan
             for m in MODEL_ORDER]
    implieds = [sub.loc[sub['model'] == m, 'implied_mult'].values[0]
                 if not sub[sub['model'] == m].empty else np.nan
                 for m in MODEL_ORDER]
    bar_cols = [MODEL_COLORS[m] for m in MODEL_ORDER]

    bars = ax.bar(range(len(MODEL_ORDER)), apes,
                  color=bar_cols, alpha=0.82, edgecolor='white', linewidth=0.8)

    valid = [(i, a) for i, a in enumerate(apes) if not np.isnan(a)]
    if valid:
        best = min(valid, key=lambda x: x[1])[0]
        bars[best].set_edgecolor('black')
        bars[best].set_linewidth(2.0)
        bars[best].set_alpha(1.0)

    for bar, imp in zip(bars, implieds):
        if not np.isnan(imp):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + max(apes) * 0.02,
                    f'{imp:.1f}x', ha='center', va='bottom',
                    fontsize=8, color='#222222')

    ax.axhline(0, color='#999999', linewidth=0.5)
    ax.set_title(f'{focal_tic}  ({role})\n{industry}  |  '
                 f'Actual = {actual:.2f}x',
                 fontsize=9.5, fontweight='bold')
    ax.set_xticks(range(len(MODEL_ORDER)))
    ax.set_xticklabels([MODEL_SHORT[m] for m in MODEL_ORDER], fontsize=8.5)
    ax.set_ylabel('APE (%)', fontsize=9)
    ax.tick_params(labelsize=8.5)

fig.suptitle('Case Study: Valuation Error by Model  '
             r'(FY2024, $k=10$, ln(EV/Sales))'
             '\nNumbers above bars = peer-implied EV/Sales. Bold border = lowest APE.',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES / 'n10_case_study_ape.pdf',
            dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print(f"Saved: {FIGURES / 'n10_case_study_ape.pdf'}")

#### 10.2 · t-SNE Visualisation of Feature Spaces

In [ ]:
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

TSNE_YEAR = 2022
N_SAMPLE  = 900
TSNE_PERP = 40
TSNE_ITER = 1000

SECTOR_MAP = {
    'Computer Software':          'Technology & Software',
    'Computers':                  'Technology & Software',
    'Electronic Equipment':       'Technology & Software',
    'Business Services':          'Technology & Software',
    'Measuring and Control Eq.':  'Technology & Software',
    'Pharmaceutical Products':    'Healthcare & Pharma',
    'Medical Equipment':          'Healthcare & Pharma',
    'Healthcare':                 'Healthcare & Pharma',
    'Banking':                    'Business & Fin. Services',
    'Insurance':                  'Business & Fin. Services',
    'Trading':                    'Business & Fin. Services',
    'Petroleum and Natural Gas':  'Energy & Industrials',
    'Utilities':                  'Energy & Industrials',
    'Machinery':                  'Energy & Industrials',
    'Chemicals':                  'Energy & Industrials',
    'Retail':                     'Consumer',
    'Food Products':              'Consumer',
    'Restaurants, Hotels, Motels':'Consumer',
    'Wholesale':                  'Consumer',
    'Apparel':                    'Consumer',
}
SECTOR_COLORS = {
    'Technology & Software':      '#4C72B0',
    'Healthcare & Pharma':        '#55A868',
    'Business & Fin. Services':   '#C44E52',
    'Energy & Industrials':       '#DD8452',
    'Consumer':                   '#8172B3',
    'Other':                      '#b0b0b0',
}

# Load feature data
df_fin = pd.read_parquet(FINANCIALS_NORM)
with open(SELECTED_FEATURES_FILE) as f:
    feat_manifest = json.load(f)
selected_features = feat_manifest['selected_features']

df_emb = pd.read_parquet(EMBEDDINGS)
emb_cols = [c for c in df_emb.columns if c.startswith('emb_')]
if not emb_cols:
    raise ValueError(f"No emb_* columns in EMBEDDINGS: {df_emb.columns.tolist()[:10]}")

print(f"Financial features : {len(selected_features)}")
print(f"Embedding dims     : {len(emb_cols)}")

# Filter to year + inner join
fin_y = (df_fin[df_fin['fyear']==TSNE_YEAR]
         [['tic'] + selected_features].dropna().copy())
emb_y = (df_emb[df_emb['fyear']==TSNE_YEAR]
         [['tic'] + emb_cols].dropna().copy())
pan_y = df_panel[df_panel['fyear']==TSNE_YEAR][['tic','industry']].copy()
common = fin_y.merge(emb_y, on='tic').merge(pan_y, on='tic')
common['sector'] = common['industry'].map(SECTOR_MAP).fillna('Other')

print(f"Common firms (FY{TSNE_YEAR}): {len(common):,}")

# Subsample
np.random.seed(RANDOM_SEED)
n = min(N_SAMPLE, len(common))
idx = np.random.choice(len(common), size=n, replace=False)
sample = common.iloc[idx].reset_index(drop=True)

fin_mat = sample[selected_features].values
emb_mat = sample[emb_cols].values
sectors = sample['sector'].values

# Run t-SNE
print("\nRunning t-SNE on financial features...")
fin_scaled = StandardScaler().fit_transform(fin_mat)
coords_fin = TSNE(n_components=2, perplexity=TSNE_PERP, max_iter=TSNE_ITER,
                   random_state=RANDOM_SEED).fit_transform(fin_scaled)

print("Running t-SNE on text embeddings...")
coords_text = TSNE(n_components=2, perplexity=TSNE_PERP, max_iter=TSNE_ITER,
                    random_state=RANDOM_SEED).fit_transform(emb_mat)
print("Done.")

In [ ]:
# Figure
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
panel_labels = [
    ('M1: Financial Ratio Space\n(64-dim -> t-SNE 2D)',  coords_fin),
    ('M2: FinBERT Text Embedding Space\n(768-dim -> t-SNE 2D)', coords_text),
]

for ax, (title, coords) in zip(axes, panel_labels):
    for sector, color in SECTOR_COLORS.items():
        mask = sectors == sector
        ax.scatter(coords[mask, 0], coords[mask, 1], c=color, s=14, alpha=0.65,
                   linewidths=0, label=sector)
    ax.set_title(title, fontsize=12, fontweight='bold', pad=10)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

handles = [mpatches.Patch(color=c, label=s) for s, c in SECTOR_COLORS.items()]
fig.legend(handles=handles, loc='lower center', ncol=3,
           fontsize=9.5, framealpha=0.9, bbox_to_anchor=(0.5, -0.06))
fig.suptitle(f't-SNE Visualisation of Peer Similarity Spaces  '
             f'(FY{TSNE_YEAR}, n = {n} firms)\n'
             'Financial space crosses industry boundaries; '
             'text space clusters more tightly by product-market category',
             fontsize=11, y=1.02)
plt.tight_layout()
save_path = FIGURES / f'n10_tsne_embedding_spaces_{TSNE_YEAR}.pdf'
plt.savefig(save_path, dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print(f"Saved: {save_path}")

In [ ]:
# t-SNE COMPANION — Quantitative sector separation metrics
# Computed on original high-dim vectors (not t-SNE reduction)
# M1: StandardScaler-normalised 64-dim financial features
# M2: L2-normalised 768-dim FinBERT embeddings

from sklearn.metrics import silhouette_samples, silhouette_score
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
sector_labels = le.fit_transform(sectors)
sector_names  = le.classes_

# ── Overall silhouette scores ─────────────────────────────────────────────────
sil_fin_overall  = silhouette_score(fin_scaled, sector_labels, metric="euclidean")
sil_text_overall = silhouette_score(emb_mat,    sector_labels, metric="euclidean")
print(f"Overall silhouette — M1 Financial : {sil_fin_overall:.4f}")
print(f"Overall silhouette — M2 Text      : {sil_text_overall:.4f}")

# ── Per-sector silhouette scores ──────────────────────────────────────────────
sil_fin_samples  = silhouette_samples(fin_scaled, sector_labels, metric="euclidean")
sil_text_samples = silhouette_samples(emb_mat,    sector_labels, metric="euclidean")

records_sil = []
for s, name in enumerate(sector_names):
    mask = sector_labels == s
    records_sil.append({
        "sector":   name,
        "n":        mask.sum(),
        "sil_fin":  sil_fin_samples[mask].mean(),
        "sil_text": sil_text_samples[mask].mean(),
    })

sil_df = pd.DataFrame(records_sil).sort_values("sil_text", ascending=False)
print("\nPer-sector silhouette scores:")
print(sil_df.to_string(index=False))

In [ ]:
# ── Per-sector within/between distance ratio ──────────────────────────────────
# ratio = mean_within / mean_between  →  lower = tighter cluster

def wb_ratio(X, labels, target_label):
    mask = labels == target_label
    X_in  = X[mask]
    X_out = X[~mask]
    if len(X_in) < 2 or len(X_out) < 2:
        return np.nan
    # sample for speed if large
    if len(X_in) > 200:
        idx = np.random.choice(len(X_in), 200, replace=False)
        X_in = X_in[idx]
    if len(X_out) > 200:
        idx = np.random.choice(len(X_out), 200, replace=False)
        X_out = X_out[idx]
    from sklearn.metrics import pairwise_distances
    within  = pairwise_distances(X_in,         metric="euclidean").mean()
    between = pairwise_distances(X_in, X_out,  metric="euclidean").mean()
    return within / between

records_wb = []
for s, name in enumerate(sector_names):
    records_wb.append({
        "sector":   name,
        "n":        (sector_labels == s).sum(),
        "wb_fin":   wb_ratio(fin_scaled, sector_labels, s),
        "wb_text":  wb_ratio(emb_mat,    sector_labels, s),
    })

wb_df = pd.DataFrame(records_wb).sort_values("wb_text")
print("Per-sector within/between ratio (lower = tighter cluster):")
print(wb_df.to_string(index=False))

In [ ]:
# ── Plot: companion figure to t-SNE (paper version, 2 panels) ────────────────
# OUTPUTS = [FIGURES / "n10_tsne_sector_separation.pdf"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8),
                          gridspec_kw={'width_ratios': [1, 1.6]})

# ── Panel 1: Overall silhouette bar ───────────────────────────────────────────
ax = axes[0]
vals   = [sil_fin_overall, sil_text_overall]
labels = ["M1: Financial\n(64-dim)", "M2: Text\n(768-dim)"]
colors = [MODEL_COLORS["M1_Financial"], MODEL_COLORS["M2_Text"]]
bars   = ax.bar(labels, vals, color=colors, width=0.55, alpha=0.88,
                edgecolor='white', linewidth=0.6)

for bar, val in zip(bars, vals):
    va  = "bottom" if val >= 0 else "top"
    y   = val + 0.004 if val >= 0 else val - 0.004
    ax.text(bar.get_x() + bar.get_width() / 2, y,
            f"{val:+.3f}", ha="center", va=va, fontsize=11,
            fontweight="bold")

ax.axhline(0, color="0.3", lw=0.8)
yabs = max(abs(sil_fin_overall), abs(sil_text_overall))
ax.set_ylim(-yabs * 1.5, yabs * 1.5)
ax.set_ylabel("Silhouette score", fontsize=10)
ax.set_title("(a) Overall sector separation", fontsize=11,
             fontweight='bold', loc="left", pad=8)
ax.tick_params(labelsize=9)
sns.despine(ax=ax)

# ── Panel 2: Per-sector silhouette grouped bar ────────────────────────────────
ax = axes[1]
x    = np.arange(len(sil_df))
w    = 0.38
b1   = ax.bar(x - w/2, sil_df["sil_fin"],  width=w,
              color=MODEL_COLORS["M1_Financial"], alpha=0.88,
              edgecolor='white', linewidth=0.6, label="M1 Financial")
b2   = ax.bar(x + w/2, sil_df["sil_text"], width=w,
              color=MODEL_COLORS["M2_Text"],      alpha=0.88,
              edgecolor='white', linewidth=0.6, label="M2 Text")

# Annotate the largest gap (Healthcare & Pharma) since the body prose
# discusses it specifically
for i, (sf, st) in enumerate(zip(sil_df["sil_fin"], sil_df["sil_text"])):
    gap = st - sf
    if abs(gap) == abs(sil_df["sil_text"] - sil_df["sil_fin"]).max():
        y_top = max(sf, st) + 0.02
        ax.annotate(f'gap = {gap:+.2f}', xy=(i, y_top), ha='center',
                    fontsize=8.5, fontstyle='italic', color='#444444')

ax.axhline(0, color="0.4", lw=0.8, ls="--")
ax.set_xticks(x)
ax.set_xticklabels(sil_df["sector"], rotation=25, ha="right", fontsize=9)
ax.set_ylabel("Mean silhouette score", fontsize=10)
ax.set_title("(b) Per-sector silhouette score", fontsize=11,
             fontweight='bold', loc="left", pad=8)
ax.legend(fontsize=9, loc='upper right', frameon=True, framealpha=0.95)
ax.tick_params(axis='y', labelsize=9)
sns.despine(ax=ax)

fig.suptitle(
    f"Sector separation in M1 Financial and M2 Text embedding spaces "
    f"(FY{TSNE_YEAR}, n = {n} firms)",
    fontsize=11.5, fontweight='bold', y=1.02
)
plt.tight_layout()
out = FIGURES / f"n10_tsne_sector_separation_{TSNE_YEAR}.pdf"
fig.savefig(out, dpi=FIGURE_DPI, format=FIGURE_FORMAT, bbox_inches="tight")
plt.show()
print(f"Saved → {out}")

### 11 · Narrative Diffusion Analysis

In [ ]:
# ── AI/digital-transformation narrative diffusion check ──────────────────────
import pandas as pd
import re

AI_LEXICON = [
    'artificial intelligence', 'machine learning', 'deep learning',
    'generative ai', 'large language model', 'llm',
    'neural network', 'ai-powered', 'ai-driven', 'ai-enabled',
    'digital transformation', 'cloud-native', 'data-driven',
    'predictive analytics', 'automation platform',
]

# Build regex with word boundaries
ai_pattern = re.compile(
    r'\b(' + '|'.join(re.escape(t) for t in AI_LEXICON) + r')\b',
    flags=re.IGNORECASE
)

# Load all summaries
summaries = []
for yr in range(2020, 2025):
    df = pd.read_csv(SUMMARIES_FILES[yr])
    df = df[df['business_description'].notna() &
            (df['business_description'].str.len() > 50)]
    df['fyear'] = yr
    summaries.append(df[['tic', 'fyear', 'business_description']])
summaries = pd.concat(summaries, ignore_index=True)

# Count AI-lexicon hits per summary
summaries['ai_hits'] = summaries['business_description'].apply(
    lambda t: len(ai_pattern.findall(t))
)
summaries['has_ai'] = summaries['ai_hits'] > 0

# Aggregate by year
ai_diffusion = (summaries
                .groupby('fyear')
                .agg(n_summaries   = ('tic',     'count'),
                     pct_with_ai   = ('has_ai',  lambda x: x.mean() * 100),
                     mean_hits     = ('ai_hits', 'mean'),
                     median_hits   = ('ai_hits', 'median'))
                .round(2))

print('AI / digital-transformation narrative diffusion in 10-K summaries:')
print(ai_diffusion.to_string())

# Optional: which terms drive the trend?
term_by_year = []
for yr in range(2020, 2025):
    sub = summaries[summaries['fyear'] == yr]['business_description']
    counts = {term: sub.str.contains(term, case=False, regex=False).sum()
              for term in AI_LEXICON}
    term_by_year.append({'fyear': yr, **counts})
term_df = pd.DataFrame(term_by_year).set_index('fyear')
print('\nTop term counts by year:')
print(term_df.T.sort_values(by=2024, ascending=False).head(8).to_string())

ai_diffusion.to_csv(DATA_RESULTS / 'ai_narrative_diffusion.csv')

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

# ── Left panel: share of summaries with AI language ──────────────────────────
ax = axes[0]
years = ai_diffusion.index.tolist()
ax.plot(years, ai_diffusion['pct_with_ai'], marker='o', linewidth=2.2,
        color='#2166ac', markersize=8)
for yr, val in zip(years, ai_diffusion['pct_with_ai']):
    ax.text(yr, val + 0.6, f'{val:.1f}%', ha='center', fontsize=9)
ax.set_xlabel('Fiscal year')
ax.set_ylabel('Share of summaries with at least one AI/digital term (%)')
ax.set_title('(a) Diffusion of AI / digital-transformation language',
             fontsize=10.5)
ax.set_xticks(years)
ax.set_ylim(0, 25)

# ── Right panel: top terms over time ─────────────────────────────────────────
ax = axes[1]
top_terms = ['artificial intelligence', 'machine learning', 'llm',
             'ai-powered', 'generative ai']
colors = ['#2166ac', '#d6604d', '#4dac26', '#8073ac', '#e08214']
for term, c in zip(top_terms, colors):
    counts = [term_df.loc[yr, term] for yr in years]
    ax.plot(years, counts, marker='o', linewidth=1.8, label=term,
            color=c, markersize=6)
ax.set_xlabel('Fiscal year')
ax.set_ylabel('Mentions across all summaries')
ax.set_title('(b) Term frequency by year', fontsize=10.5)
ax.legend(fontsize=8, loc='upper left')
ax.set_xticks(years)

fig.suptitle('AI and digital-transformation narrative diffusion in 10-K Item~1 summaries',
             fontsize=11.5, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES / 'ai_narrative_diffusion.pdf', dpi=FIGURE_DPI,
            bbox_inches='tight')
plt.show()

### 12 · Save

In [ ]:
results_df['run_date'] = pd.Timestamp.now().strftime('%Y-%m-%d')
results_df['status']   = 'complete'
results_df.to_csv(RESULTS_MAIN, index=False)
print(f"Saved: {RESULTS_MAIN}")
print(f"Rows : {len(results_df)}")

In [ ]:
# Final summary print
print("=" * 70)
print("FINAL RESULTS SUMMARY")
print("=" * 70)
print(f"Evaluation sample: {len(summary_tickers):,} firm-years  "
      f"({multiples['tic'].nunique():,} unique firms)")
print(f"Primary metric  : MdAPE on ln(EV/Sales), k={K_MAIN}")
print(f"Validation years: {VALIDATION_YEARS}")
print(f"Test years      : {TEST_YEARS}")
print()

for model_name in ['M0_FF49','M1_Financial','M2_Text','M3_Fusion']:
    row = results_df[(results_df['model']==model_name) &
                     (results_df['multiple']=='ln_v2s') &
                     (results_df['k']==K_MAIN)].iloc[0]
    print(f"  {MODEL_LABELS[model_name]:<28} "
          f"MdAPE={row['mdape']*100:.2f}%  "
          f"[{row['ci_lo']*100:.2f}%, {row['ci_hi']*100:.2f}%]")

m0 = results_df[(results_df['model']=='M0_FF49') &
                (results_df['multiple']=='ln_v2s') &
                (results_df['k']==K_MAIN)]['mdape'].values[0]
m3 = results_df[(results_df['model']=='M3_Fusion') &
                (results_df['multiple']=='ln_v2s') &
                (results_df['k']==K_MAIN)]['mdape'].values[0]

print(f"\n  Total improvement M0 -> M3 : {(m0-m3)/m0*100:+.1f}%")
print(f"  Optimal alpha (M3)         : {alpha_info['best_alpha']}")
print(f"  Fusion method              : Weighted Rank (+ RRF robustness check)")
print(f"\nAll hypothesis tests (H1, H2, H3): see Section 4.")
print(f"Sector-level diagnostics (Section 6) support discussion only; "
      f"no formal H4 test.")